# 🕰️ Git restore point

The version of this notebook immediately before the CACO3 functionality was added is preserved in Git commit `d471644` (`CandidateParameterSetsBio before CACO3 functionality`).

To restore only this notebook to that exact version, run the following command from the repository directory:

```bash
git restore --source=d471644 -- CandidateParameterSetsBio.ipynb
```

<div class="alert alert-danger"><strong>Restore warning:</strong> This command replaces the notebook in the working tree. Commit any later work first if it also needs to be preserved.</div>

# 🧭 CandidateParameterSetsBio workflow

This notebook turns completed ROMS runs into an Optuna study and, when explicitly requested, creates the next batch of candidate input files. Run the notebook from top to bottom; later sections depend on objects created earlier.

1. **Configure the run.** Read file locations, scientific settings, CPP options, study settings, and safety switches.
2. **Define helper functions.** Set up cost, parameter, CPP-option, date, and advection extraction.
3. **Read completed runs.** Open each cost and station file once, calculate diagnostics and total cost, and collect model parameters.
4. **Validate and filter.** Report file/data problems, remove unusable or high-cost runs, and prepare the Optuna dataframe.
5. **Load the Optuna study.** Create a new SQLite study when the database is absent or load the existing study when it is present.
6. **Register results.** Add historical completed trials and optionally complete previously generated `RUNNING` trials whose model output is now available.
7. **Generate candidates.** Optionally ask Optuna for new parameter sets.
8. **Build candidate files.** Optionally apply candidate values to water-column and sediment templates and write validated input files.
9. **Review and export.** Report final study status and optionally export the cleaned run summary.

<div class="alert alert-info"><strong>Safety default:</strong> Database completion, candidate generation, candidate-file writing, and summary export are all off unless their individual switches are deliberately enabled. Historical trials are still imported into a newly created study.</div>

# ▶️ How to use this notebook

1. Select the **`my_env` Python environment** as the notebook kernel. It contains the scientific and Optuna packages used here.
2. Search for **`USER INPUT`** before every run. Every setting that may require a decision or path change is marked with that exact phrase.
3. Start with the **required paths and identifiers** in Section 1. Confirm that `FILE_LIST`, `PARAMETER_LIST`, and `RUN_MAP_FILE` exist and use the expected columns.
4. Review the **scientific configuration**: included targets, cost threshold, cost weights, baseline run, overrides, CPP options, active Optuna parameters, and parameter bounds.
5. Review the four **action switches**. Leave them `False` for a read/validation pass. Enable only the action you intend:
   - `UPDATE_OPTUNA_WITH_COMPLETED_RUNS`: finish matching `RUNNING` trials in the database.
   - `GENERATE_NEW_CANDIDATES`: create a new batch only when no trials remain `RUNNING`.
   - `WRITE_CANDIDATE_INPUT_FILES`: write files for available `RUNNING` candidates.
   - `EXPORT_RUN_SUMMARY`: write the cleaned analysis table.
6. Run all cells from the top. Stop and investigate any raised exception or warning summary before using generated candidates.
7. Review the historical-import counts, completion preview, candidate table, parameter-source audit, and final study status.
8. If the result is wrong, restore the pre-CACO3 notebook with the Git command in the first block.

<div class="alert alert-warning"><strong>Important limitation:</strong> CPP states such as <code>CACO3</code> are recorded for historical analysis but are not yet applied when candidate input files are written. See the limitation in Section 7.</div>

# 🗺️ Notebook map

Use these links to jump between the notebook's major stages. The intended execution order is still top to bottom.

| Stage | Purpose |
| :--- | :--- |
| [⚙️ 1. Configuration](#configuration) | Review paths, science settings, study identity, and action switches. |
| [🧰 2. Functions](#functions) | Define reusable extraction, diagnostic, and replacement helpers. |
| [📥 3. Read completed runs](#read-runs) | Calculate diagnostics and collect station-file parameters. |
| [🧹 4. Prepare data](#prepare-data) | Validate, filter, and build the Optuna dataframe. |
| [🧠 5. Optuna study](#optuna-study) | Load the study and register historical trials. |
| [🚀 6. Candidates](#candidates) | Synchronize finished runs and optionally request candidates. |
| [📝 7. Candidate files](#candidate-files) | Audit values, generate text, and optionally write input files. |
| [✅ 8. Final status](#final-status) | Review trial states and optionally export the run summary. |

**Color key:** 🔵 information · 🟡 caution or limitation · 🔴 database/file change · 🟢 ready or safe default

# 🛫 Pre-run checklist

Treat this as a static checklist to review before choosing **Run All**.

- ☐ The `my_env` Python kernel is selected.
- ☐ The cost, optimization, workbook, parameter-list, and run-map paths are correct.
- ☐ The target selection, threshold, cost weights, baseline, and overrides are intentional.
- ☐ The active numeric parameters appear in the valid-bounds table.
- ☐ The active categorical parameters use only the printed choices.
- ☐ The Optuna study name and database file are correct.
- ☐ Every action switch has been reviewed; uncertain switches remain `False`.
- ☐ Template paths and output locations are correct before enabling file writing.

<div class="alert alert-success"><strong>Ready:</strong> If every applicable item above has been reviewed, run the notebook from the top and pause at any raised error or warning summary.</div>

In [1]:
# Basic packages
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd
import math
from netCDF4 import Dataset, num2date

# DateTime packages
from matplotlib.dates import DateFormatter
from datetime import datetime, timedelta
import time
import matplotlib.dates as mdates

# Stats packages
import scipy
import PyCO2SYS as pyco2
import gsw
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
import optuna

# Logistical packages
import requests
from importlib import reload
import warnings
import re
from pathlib import Path
import gc

<a id="configuration"></a>
# ⚙️ 1. Configuration

This biological calibration notebook excludes temperature and salinity from both the diagnostic metrics and total cost. It adds two station-file predictors to the existing biological parameter suite: `nl_tnu2_tracer9` and the paired horizontal/vertical advection scheme reported for `detritus` in `NLM_TADV`. It also records configured CPP-option states and their conditional station-file parameters for historical trial analysis.

Tracer 9 is used as a representative of all biological tracers. When candidate files are written, its suggested `nl_tnu2` and advection pair are applied uniformly to all 15 biological tracers. Database updates, new candidate generation, file writing, and run-summary export all default to off.

> Settings that may require review or editing are marked `USER INPUT`. Numbered labels identify the main configuration groups; `(advanced)` settings should usually remain unchanged.

In [2]:
# ============================================================
# USER INPUT 1: FILE AND DIRECTORY PATHS
# ============================================================

# >>> USER INPUT: Confirm these locations before running the notebook.
COST_DIR = Path("/Users/akbaskind/Desktop/COST_FILES")
OPT_DIR = Path("/Users/akbaskind/Desktop/Optimization")
FILE_LIST = OPT_DIR / "FileNames.xlsx"
PARAMETER_LIST = OPT_DIR / "ParameterList copy.csv"
RUN_MAP_FILE = OPT_DIR / "runmap.xlsx"

# >>> USER INPUT (advanced): Update these names only if the station-file
# schema or desired fallback date changes.
DSTART_VARIABLE = "dstart"
DSTART_YEAR_COLUMN = "dstart_year"
HISTORY_ATTRIBUTE = "history"
RUN_DATE_COLUMN = "Run Date"
RUN_DATE_FALLBACK = pd.Timestamp("2000-01-01")

# ============================================================
# USER INPUT 2: RECORDED CPP OPTIONS AND CONDITIONAL PARAMETERS
# ============================================================

# >>> USER INPUT: Add future CPP options here when they should be stored
# as Boolean categorical trial parameters. Each configured option is read
# from the station file's CPP_options
# global attribute and stored as a Boolean categorical Optuna parameter.
# Future options that need no conditional numeric values can be added here.
CPP_OPTIONS_ATTRIBUTE = "CPP_options"
CPP_OPTION_PARAMETERS = ("CACO3",)

# >>> USER INPUT: For any option with conditional numeric variables, map
# each station-file variable to its resulting scalar column name(s).
# Map each option to its station-file variables and resulting scalar column
# names. Array-valued variables such as bUmaxCa are expanded explicitly.
CPP_CONDITIONAL_FLOAT_VARIABLES = {
    "CACO3": {
        "cacopf": ("cacopf",),
        "cacodr": ("cacodr",),
        "omega_thresh": ("omega_thresh",),
        "wsPCa": ("wsPCa",),
        "bUmaxCa": (
            "bUmaxCa_nspc0",
            "bUmaxCa_nspc1",
            "bUmaxCa_nspc2",
        ),
    },
}

CPP_RECORDED_FLOAT_PARAMETERS = tuple(
    output_name
    for option_name in CPP_OPTION_PARAMETERS
    for output_names in CPP_CONDITIONAL_FLOAT_VARIABLES.get(
        option_name, {}
    ).values()
    for output_name in output_names
)
CPP_CONDITIONAL_STATION_VARIABLES = tuple(
    variable_name
    for option_name in CPP_OPTION_PARAMETERS
    for variable_name in CPP_CONDITIONAL_FLOAT_VARIABLES.get(
        option_name, {}
    )
)
CPP_RECORDED_PARAMETERS = (
    *CPP_OPTION_PARAMETERS,
    *CPP_RECORDED_FLOAT_PARAMETERS,
)

# Load the run inventory and the standard station-parameter list.
# >>> USER INPUT (advanced): Spreadsheet column names expected by the code.
required_run_columns = ["Run Name", "Cost File", "Station File"]
runs = pd.read_excel(FILE_LIST)
runs = runs[required_run_columns].dropna(
    subset=required_run_columns
)

parameter_names = (
    pd.read_csv(PARAMETER_LIST)["Parameter Name"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

print(f"{len(runs)} runs")
print(f"{len(parameter_names)} parameters")

# ============================================================
# USER INPUT 3: TARGET SELECTION
# ============================================================

# >>> USER INPUT: Choose whether benthic metrics contribute to diagnostics
# and cost. exclude_some_benthic applies only when include_benthic is True.
include_benthic = True
exclude_some_benthic = True

benthic_cols = [
    "SOD",
    "Benthic NO3 Flux",
    "Benthic NH4 Flux",
]
some_benthic_cols = [
    "Benthic NO3 Flux",
    "Benthic NH4 Flux",
]

# ============================================================
# USER INPUT 4: FILTERS AND THRESHOLDS
# ============================================================

# >>> USER INPUT: Runs at or above this cost are excluded from Optuna.
COST_THRESHOLD = 200

# ============================================================
# USER INPUT 5: METRIC DEFINITIONS FOR nRMSE
# ============================================================

# >>> USER INPUT (advanced): Edit only when model/observation variable
# names or diagnostic targets change.
metric_info = {
    "Surface pH": ("pH_mod", "pH_obs"),
    "Bottom pH": ("pH_mod", "pH_obs"),
    # "Total pH":   ("pH_mod", "pH_obs"),

    "Surface Oxygen": ("oxygen_mod", "oxygen_obs"),
    "Bottom Oxygen":  ("oxygen_mod", "oxygen_obs"),
    # "Total Oxygen":   ("oxygen_mod", "oxygen_obs"),

    "Surface NO3": ("NO3_mod", "NO3_obs"),
    "Bottom NO3":  ("NO3_mod", "NO3_obs"),
    # "Total NO3":   ("NO3_mod", "NO3_obs"),

    "Surface NH4": ("NH4_mod", "NH4_obs"),
    "Bottom NH4":  ("NH4_mod", "NH4_obs"),
    # "Total NH4":   ("NH4_mod", "NH4_obs"),

    "Surface Si": ("Si_mod", "Si_obs"),
    "Bottom Si":  ("Si_mod", "Si_obs"),
    # "Total Si":   ("Si_mod", "Si_obs"),

    "Secchi Depth": ("SD_mod", "SD_obs"),

    "SOD": ("sed_o2_mod", "sed_o2_obs"),
    "Benthic NO3 Flux": ("sed_no3_mod", "sed_no3_obs"),
    "Benthic NH4 Flux": ("sed_nh4_mod", "sed_nh4_obs"),
}

if not include_benthic:
    metric_info = dict([i for i in metric_info.items() if i[0] not in benthic_cols])
elif exclude_some_benthic:
    metric_info = dict([i for i in metric_info.items() if i[0] not in some_benthic_cols])


print(f"\nNumber of nRMSE metrics = {len(metric_info)}")
print(f"nRMSE metrics: {list(metric_info.keys())}")

# ============================================================
# USER INPUT 6: METRIC DEFINITIONS AND WEIGHTS FOR COST
# ============================================================

# >>> USER INPUT (advanced): Keep target names synchronized among
# metric_info, cost_components, and cost_weights.
cost_components = {
    "Surface pH": "pH_cost",
    "Bottom pH": "pH_cost",
    "Surface Oxygen": "oxygen_cost",
    "Bottom Oxygen": "oxygen_cost",
    "Surface NO3": "NO3_cost",
    "Bottom NO3": "NO3_cost",
    "Surface NH4": "NH4_cost",
    "Bottom NH4": "NH4_cost",
    "Surface Si": "Si_cost",
    "Bottom Si": "Si_cost",
    "Secchi Depth": "SD_cost",
    "SOD": "sed_o2_cost",
    "Benthic NO3 Flux": "sed_no3_cost",
    "Benthic NH4 Flux": "sed_nh4_cost",
}

cost_weights = {
    "Surface pH": 2,
    "Bottom pH": 2,
    "Surface Oxygen": 2,
    "Bottom Oxygen": 2,
    "Surface NO3": 1,
    "Bottom NO3": 1,
    "Surface NH4": 1,
    "Bottom NH4": 1,
    "Surface Si": 1,
    "Bottom Si": 1,
    "Secchi Depth": 1,
    "SOD": 1,
    "Benthic NO3 Flux": 1,
    "Benthic NH4 Flux": 1,
}

if not include_benthic:
    cost_components = dict([i for i in cost_components.items() if i[0] not in benthic_cols])
    cost_weights = dict([i for i in cost_weights.items() if i[0] not in benthic_cols])
elif exclude_some_benthic:
    cost_components = dict([i for i in cost_components.items() if i[0] not in some_benthic_cols])
    cost_weights = dict([i for i in cost_weights.items() if i[0] not in some_benthic_cols])

print(f"\nNumber of cost targets = {len(cost_components)}")
print(f"Cost metrics: {list(cost_components.keys())}")

# ============================================================
# USER INPUT 7: CANDIDATE BASELINE AND SCIENTIFIC OVERRIDES
# ============================================================

# >>> USER INPUT: This run supplies fixed values for parameters that Optuna
# is not varying. The run must survive the cost filter and be unique.
BASELINE_RUN_NAME = "DUMar26"

# >>> USER INPUT: These values take precedence over the selected baseline
# but do not modify the original template files.
SCIENTIFIC_OVERRIDES = {
    "bao2": 2.0,
}

# ============================================================
# USER INPUT 8: REPRESENTATIVE BIOLOGICAL TRACER SETTINGS
# ============================================================

# >>> USER INPUT (advanced): Change only if a different biological tracer
# should represent mixing and advection for all biological tracers.
BIO_TRACER_INDEX = 9
BIO_TRACER_ATTRIBUTE_NAME = "detritus"
BIO_NL_TNU2_PARAMETER = "nl_tnu2_tracer9"
BIO_ADVECTION_PARAMETER = "tracer9_advection_scheme"
BIOLOGICAL_TRACER_COUNT = 15

# >>> USER INPUT (advanced): Keep categories and input codes synchronized
# with the advection names supported by ROMS and the selected templates.
# Categories use descriptive station-file names. Input-file aliases are
# applied only when candidate text is generated.
BIO_ADVECTION_SCHEMES = {
    "Upstream3_Centered4": ("Upstream3", "Centered4"),
    "HSIMT": ("HSIMT", "HSIMT"),
    "Akima4": ("Akima4", "Akima4"),
    "MPDATA": ("MPDATA", "MPDATA"),
}
ADVECTION_INPUT_CODES = {
    "Upstream3": "U3",
    "Centered4": "C4",
    "Akima4": "A4",
    "MPDATA": "MPDATA",
    "HSIMT": "HSIMT",
}

# ============================================================
# USER INPUT 9: OPTUNA STUDY IDENTITY AND STORAGE
# ============================================================

# >>> USER INPUT: Use a new study name/database when the objective or search
# space should not be combined with the existing study.
# Separate study because this cost excludes temperature and salinity and
# includes representative biological mixing/advection parameters.
STUDY_NAME = "model_calibration_bio"
CURRENT_STUDY_NAME = STUDY_NAME

STORAGE_FILE = OPT_DIR / "model_calibration_bio.db"

STORAGE = f"sqlite:///{STORAGE_FILE}"

# >>> USER INPUT (advanced): Required run-map spreadsheet columns.
# Load and validate the shared mapping from study-assigned names to model run names.
required_run_map_columns = [
    "Actual Run Name", "Study Name", "Study Run Name", "Trial Number"
]
run_map = pd.read_excel(RUN_MAP_FILE)
run_map.columns = run_map.columns.astype(str).str.strip()
missing_run_map_columns = [
    column for column in required_run_map_columns if column not in run_map.columns
]
if missing_run_map_columns:
    raise ValueError(
        f"Run map is missing required columns: {missing_run_map_columns}"
    )

for column in ["Actual Run Name", "Study Name", "Study Run Name"]:
    run_map[column] = run_map[column].astype("string").str.strip()

trial_numbers = pd.to_numeric(run_map["Trial Number"], errors="coerce")
invalid_trial_numbers = trial_numbers.isna() | (trial_numbers % 1 != 0)
if invalid_trial_numbers.any():
    invalid_rows = (run_map.index[invalid_trial_numbers] + 2).tolist()
    raise ValueError(
        f"Run map has invalid Trial Number values in Excel rows: {invalid_rows}"
    )
run_map["Trial Number"] = trial_numbers.astype(int)

current_study_run_map = run_map.loc[
    run_map["Study Name"].eq(CURRENT_STUDY_NAME)
].copy()
duplicate_mapping_keys = [
    ["Study Name", "Trial Number"],
    ["Study Name", "Study Run Name"],
    ["Study Name", "Actual Run Name"],
]
duplicate_mapping_messages = []
for columns in duplicate_mapping_keys:
    duplicate_rows = current_study_run_map.loc[
        current_study_run_map.duplicated(columns, keep=False), columns
    ]
    if not duplicate_rows.empty:
        duplicate_mapping_messages.append(
            f"{columns}: {duplicate_rows.to_dict('records')}"
        )
if duplicate_mapping_messages:
    raise ValueError(
        f"Invalid duplicate run-map mappings for {CURRENT_STUDY_NAME!r}:\n"
        + "\n".join(duplicate_mapping_messages)
    )

print(f"\nStudy name: {STUDY_NAME}")
print(f"Storage:    {STORAGE_FILE}")
print(f"Run-map rows for this study: {len(current_study_run_map)}")

# ============================================================
# USER INPUT 10: ACTION SWITCHES (ALL DEFAULT TO OFF)
# ============================================================

# >>> USER INPUT: Set True to tell Optuna about RUNNING trials whose model results now exist.
# When False, eligible trials are reported but the study is not modified.
UPDATE_OPTUNA_WITH_COMPLETED_RUNS = False

# >>> USER INPUT: Set True only when a new batch of suggestions is needed.
GENERATE_NEW_CANDIDATES = False
# >>> USER INPUT: Number of new candidates requested when generation is on.
N_CANDIDATES = 4

# >>> USER INPUT: This is independent of candidate generation, allowing files for existing
# RUNNING candidates to be regenerated without asking Optuna for new trials.
WRITE_CANDIDATE_INPUT_FILES = False

# >>> USER INPUT: Export the cleaned model-run table used downstream.
EXPORT_RUN_SUMMARY = True

# >>> USER INPUT (optional): Limit detailed mismatch locations per metric.
MAX_MISMATCH_LOCATIONS = 10

# >>> USER INPUT (advanced): Change this whenever the objective definition
# changes. It prevents incompatible results from sharing one study.
OBJECTIVE_VERSION = "cost_v3_bio_no_temp_salt"

113 runs
74 parameters

Number of nRMSE metrics = 12
nRMSE metrics: ['Surface pH', 'Bottom pH', 'Surface Oxygen', 'Bottom Oxygen', 'Surface NO3', 'Bottom NO3', 'Surface NH4', 'Bottom NH4', 'Surface Si', 'Bottom Si', 'Secchi Depth', 'SOD']

Number of cost targets = 12
Cost metrics: ['Surface pH', 'Bottom pH', 'Surface Oxygen', 'Bottom Oxygen', 'Surface NO3', 'Bottom NO3', 'Surface NH4', 'Bottom NH4', 'Surface Si', 'Bottom Si', 'Secchi Depth', 'SOD']

Study name: model_calibration_bio
Storage:    /Users/akbaskind/Desktop/Optimization/model_calibration_bio.db
Run-map rows for this study: 0


## 🎛️ Configuration snapshot

This read-only summary puts the most important choices in one place. Review it before the notebook starts processing large files. An **ON** action is highlighted because it can change the database or write files.

In [3]:
# Summarize configuration only; this cell does not read model files or write data.
DF_configuration_summary = pd.DataFrame([
    {"Setting": "Runs listed", "Value": len(runs)},
    {"Setting": "Station parameters listed", "Value": len(parameter_names)},
    {"Setting": "Cost threshold", "Value": COST_THRESHOLD},
    {"Setting": "Baseline run", "Value": BASELINE_RUN_NAME},
    {"Setting": "Study name", "Value": STUDY_NAME},
    {"Setting": "Database", "Value": str(STORAGE_FILE)},
    {"Setting": "Objective version", "Value": OBJECTIVE_VERSION},
    {"Setting": "Recorded CPP options", "Value": ", ".join(CPP_OPTION_PARAMETERS)},
])

DF_action_switches = pd.DataFrame([
    {"Action": "Complete RUNNING trials", "Enabled": UPDATE_OPTUNA_WITH_COMPLETED_RUNS},
    {"Action": "Generate candidates", "Enabled": GENERATE_NEW_CANDIDATES},
    {"Action": "Write candidate files", "Enabled": WRITE_CANDIDATE_INPUT_FILES},
    {"Action": "Export run summary", "Enabled": EXPORT_RUN_SUMMARY},
])
DF_action_switches["State"] = DF_action_switches["Enabled"].map(
    {False: "OFF", True: "ON"}
)

def color_action_state(value):
    """Highlight enabled write actions and safe disabled defaults."""

    if value == "ON":
        return "background-color: #f8d7da; color: #842029; font-weight: bold"
    return "background-color: #d1e7dd; color: #0f5132; font-weight: bold"

display(DF_configuration_summary.style.hide(axis="index"))
display(
    DF_action_switches[["Action", "State"]]
    .style.hide(axis="index")
    .map(color_action_state, subset=["State"])
)

Setting,Value
Runs listed,113
Station parameters listed,74
Cost threshold,200
Baseline run,DUMar26
Study name,model_calibration_bio
Database,/Users/akbaskind/Desktop/Optimization/model_calibration_bio.db
Objective version,cost_v3_bio_no_temp_salt
Recorded CPP options,CACO3


Action,State
Complete RUNNING trials,OFF
Generate candidates,OFF
Write candidate files,OFF
Export run summary,ON


<a id="functions"></a>
# 🧰 2. Functions

## 2.1. nRMSE functions

$$
RMSD^{*'} = \text{sign}(\Delta \sigma)\sqrt{1 + {(\frac{\sigma_a}{\sigma_{\widehat{a}}})}^2 -2(\frac{\sigma_a}{\sigma_{\widehat{a}}}) \frac{\frac{1}{N} \sum_{n=1}^{N} (a_n - \overline{a})(\widehat{a_n} - \overline{\widehat{a}})}{\sigma_a \sigma_{\widehat{a}}}}
$$

* $\widehat{a}$: observations
* $a$: model results

From [Jolliff et al. (2009)](https://doi.org/10.1016/j.jmarsys.2008.05.014).

Model and observation values are expected to share dimensions and missing-data locations. Each metric is calculated from paired finite values only. Any mismatch is printed with counts and coordinate locations (up to `MAX_MISMATCH_LOCATIONS`) and retained in a diagnostics table.

In [4]:
# ============================================================
# OBSERVATIONAL ERROR
# ============================================================

def get_stderr(var_obs, obs_mean):

    if var_obs.startswith("sed"):
        return obs_mean

    elif var_obs.startswith("pH"):
        return 0.1

    elif var_obs.startswith("temp"):
        return 0.01

    elif var_obs.startswith("salt"):
        return 0.005 * obs_mean

    elif var_obs.startswith("oxy"):
        return 3.125

    elif var_obs.startswith("N"):
        return 0.1

    elif var_obs.startswith("Si"):
        return 0.1

    elif var_obs.startswith("SD"):
        return 0.2

    else:
        return 0


# ============================================================
# CALCULATE ONE RMSE*
# ============================================================

def describe_locations(data_array, mask, limit):
    """Return readable coordinate labels for True entries in a mask."""

    locations = []

    for index_values in np.argwhere(mask)[:limit]:
        labels = []

        for dim, index in zip(data_array.dims, index_values):
            if dim in data_array.coords and data_array.coords[dim].ndim == 1:
                coordinate = data_array.coords[dim].values[index]
                labels.append(f"{dim}={coordinate}")
            else:
                labels.append(f"{dim}[{index}]")

        locations.append(", ".join(labels) if labels else "scalar")

    return locations


def record_rmse_issue(
    diagnostics,
    run_name,
    metric_name,
    issue,
    details,
    print_details=True,
):
    """Print and retain one RMSE data-quality issue."""

    record = {
        "Run Name": run_name,
        "Metric": metric_name,
        "Issue": issue,
        "Details": details,
    }
    diagnostics.append(record)
    if print_details:
        print(f"  RMSE WARNING [{metric_name}] {issue}: {details}")


def calculate_rmse_star(
    ds,
    metric_name,
    var_mod,
    var_obs,
    run_name,
    diagnostics,
    max_mismatch_locations=10,
):
    """Calculate signed RMSD*' from paired, finite values."""

    missing_variables = [
        variable
        for variable in (var_mod, var_obs)
        if variable not in ds.variables
    ]

    if missing_variables:
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Missing variable",
            f"not found: {missing_variables}",
        )
        return np.nan

    mod_data = ds[var_mod]
    obs_data = ds[var_obs]

    if metric_name.startswith("Surface"):
        mod_data = mod_data.isel(Depth=0)
        obs_data = obs_data.isel(Depth=0)
    elif metric_name.startswith("Bottom"):
        mod_data = mod_data.isel(Depth=1)
        obs_data = obs_data.isel(Depth=1)

    # Pairing is meaningful only when dimensions and shapes agree exactly.
    if mod_data.dims != obs_data.dims or mod_data.shape != obs_data.shape:
        details = (
            f"model dims/shape={mod_data.dims}/{mod_data.shape}; "
            f"observation dims/shape={obs_data.dims}/{obs_data.shape}"
        )
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Alignment mismatch", details
        )
        return np.nan

    mod_values = np.asarray(mod_data.values)
    obs_values = np.asarray(obs_data.values)
    mod_finite = np.isfinite(mod_values)
    obs_finite = np.isfinite(obs_values)

    model_only = mod_finite & ~obs_finite
    observation_only = ~mod_finite & obs_finite

    if model_only.any() or observation_only.any():
        model_locations = describe_locations(
            mod_data, model_only, max_mismatch_locations
        )
        observation_locations = describe_locations(
            obs_data, observation_only, max_mismatch_locations
        )
        details = (
            f"model finite/observation non-finite={model_only.sum()} "
            f"at {model_locations}; "
            f"observation finite/model non-finite={observation_only.sum()} "
            f"at {observation_locations}"
        )
        record_rmse_issue(
            diagnostics,
            run_name,
            metric_name,
            "Missing-data mismatch",
            details,
            print_details=False,
        )
        print(
            f"  RMSE WARNING [{metric_name}] missing-data mismatch: "
            f"model-only={model_only.sum()}, "
            f"observation-only={observation_only.sum()}"
        )

    # Use the same paired sample for every statistic.
    paired = mod_finite & obs_finite
    mod = mod_values[paired].astype(float)
    obs = obs_values[paired].astype(float)

    if mod.size < 2:
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Insufficient paired data",
            f"found {mod.size} paired finite value(s); at least 2 are required",
        )
        return np.nan

    mod_mean = mod.mean()
    obs_mean = obs.mean()
    std_mod = mod.std()
    std_obs_sample = obs.std()

    # Include the configured observational uncertainty in observation spread.
    stderr = get_stderr(var_obs, obs_mean)
    std_obs = np.sqrt(std_obs_sample**2 + stderr**2)

    if std_mod == 0 or not np.isfinite(std_mod):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid model variance",
            f"model standard deviation={std_mod}",
        )
        return np.nan

    if std_obs == 0 or not np.isfinite(std_obs):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid observation variance",
            f"adjusted observation standard deviation={std_obs}",
        )
        return np.nan

    covariance = np.mean((mod - mod_mean) * (obs - obs_mean))
    correlation = covariance / (std_mod * std_obs)
    std_norm = std_mod / std_obs
    sign = np.sign(std_mod - std_obs)

    inside = 1 + std_norm**2 - 2 * std_norm * correlation

    # Roundoff can produce a tiny negative value; a larger one is not valid.
    if inside < -1e-12 or not np.isfinite(inside):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid RMSD expression",
            f"value under square root={inside}",
        )
        return np.nan

    rmsd_star = np.sqrt(max(inside, 0.0)) * sign
    return float(rmsd_star)

## 2.2. Cost functions

The cost function is based on that found in Equation 1 of [Ward et al. (2010)](https://www.sciencedirect.com/science/article/pii/S0924796309003431?casa_token=S1tQ2yuwGQgAAAAA:lCpjjVEqP4bJWA4xg5kOjG028PuyDQdcOHfTw5KtVLoM0kkonEa15ZinoxIopsWH1nTmhNRExw).

$$
J = \frac{1}{M} \sum_{m=1}^{M} W_m^2 \frac{1}{N_m} \sum_{n=1}^{N_m} (a - \hat{a})_{n,m}^2
$$

$M$: number of data targets \
$N_m$: number of observations for each target \
$\hat{a}$: observed value of data target $m$ at location/time $n$ \
$a$: model equivalent of $\hat{a}$ \
$W_m$: weight function; $W_m = \frac{C_m}{\sigma_m}$

**Implementation notes:** The $\frac{1}{M}$ term is omitted, so comparisons assume every simulation uses the same targets. Depth-resolved cost variables use index 0 for the surface target and index 1 for the bottom target.

In [5]:
def calculate_cost(ds, cost_components, cost_weights):
    """Sum weighted surface, bottom, and non-depth-specific costs."""

    total_cost = 0.0

    for target, variable_name in cost_components.items():
        weight = cost_weights[target]

        if target.startswith("Surface"):
            component = float(ds[variable_name][0])
        elif target.startswith("Bottom"):
            component = float(ds[variable_name][1])
        else:
            component = float(ds[variable_name])

        total_cost += component * weight**2

    return total_cost

## 2.3. Parameter extraction functions

Method to extract parameter values from the station file. The `dstart` time variable is decoded separately and retained as the numeric parameter `dstart_year`.

In [6]:
def get_parameter_values(ds, parameter_name):

    # Parameter doesn't exist in this file
    # Don't create a new scalar-named column
    if parameter_name not in ds.variables:
        return {}

    var = ds.variables[parameter_name]

    try:

        # --------------------------------
        # Scalar parameter
        # --------------------------------
        if var.ndim == 0:

            value = var[...]

            if np.ma.is_masked(value):
                value = np.nan

            return {
                parameter_name: np.asarray(value).item()
            }

        # --------------------------------
        # Parameter with dimensions
        # --------------------------------
        # Copy the NetCDF-backed array immediately. Some HDF/NetCDF builds
        # reuse read buffers, which can otherwise change earlier values when
        # another one-dimensional station variable is read.
        values = np.array(var[:], copy=True).squeeze()

        # Squeezes down to a scalar
        if values.ndim == 0:

            return {
                parameter_name: values.item()
            }

        # --------------------------------
        # One-dimensional parameter
        # --------------------------------
        if values.ndim == 1:

            dim_name = var.dimensions[0]

            return {
                f"{parameter_name}_{dim_name}{i}": value
                for i, value in enumerate(values)
            }

        # --------------------------------
        # Anything else
        # --------------------------------
        values = values.ravel()

        return {
            f"{parameter_name}_{i}": value
            for i, value in enumerate(values)
        }

    except Exception as e:

        print(
            f"    Could not read {parameter_name}: {e}"
        )

        return {}


def get_dstart_year(ds, variable_name=DSTART_VARIABLE):
    """Decode the station-file start date and return only its year."""

    if variable_name not in ds.variables:
        return np.nan

    var = ds.variables[variable_name]
    raw_value = var[...]

    if np.ma.is_masked(raw_value):
        return np.nan

    value = np.asarray(raw_value).squeeze()
    if value.size != 1:
        return np.nan

    if np.issubdtype(value.dtype, np.datetime64):
        timestamp = pd.to_datetime(value.reshape(1))[0]
    else:
        units = getattr(var, "units", None)
        if units is None:
            raise ValueError(f"{variable_name} has no NetCDF time units")
        timestamp = num2date(
            value.item(),
            units=units,
            calendar=getattr(var, "calendar", "standard"),
            only_use_cftime_datetimes=False,
        )

    return int(timestamp.year)


def get_run_date(ds, attribute_name=HISTORY_ATTRIBUTE):
    """Extract the model run date from the station-file history attribute."""

    # ROMS history strings contain a date such as 'January 18, 2026'.
    # Return the documented fallback date when the attribute is absent or malformed.
    history = getattr(ds, attribute_name, None)
    if history is None:
        return RUN_DATE_FALLBACK, True

    date_match = re.search(
        r"\b(?:January|February|March|April|May|June|July|August|"
        r"September|October|November|December)\s+\d{1,2},\s+\d{4}\b",
        str(history),
    )
    if date_match is None:
        return RUN_DATE_FALLBACK, True

    run_date = pd.to_datetime(
        date_match.group(0),
        format="%B %d, %Y",
        errors="coerce",
    )
    if pd.isna(run_date):
        return RUN_DATE_FALLBACK, True

    return run_date.normalize(), False

## 2.4. CPP-option and conditional-parameter extraction

Configured CPP options are read from the station file's `CPP_options` global attribute and retained as Boolean categorical parameters. Matching uses complete CPP tokens rather than comma splitting because some ROMS option strings omit commas between adjacent options. An option is off when it is absent, explicitly prefixed with `!`, or the entire attribute is absent.

Conditional numeric variables are read only when their associated option is on. When the option is off, their recorded values are set to `0.0` without looking for the variables in the station file. Add future categorical CPP options to `CPP_OPTION_PARAMETERS`; add an entry to `CPP_CONDITIONAL_FLOAT_VARIABLES` only when an option also controls numeric station-file variables.

In [7]:
def get_cpp_option_states(
    ds,
    option_names=CPP_OPTION_PARAMETERS,
    attribute_name=CPP_OPTIONS_ATTRIBUTE,
):
    """Return configured CPP options as Boolean enabled/disabled states."""

    if attribute_name in ds.ncattrs():
        attribute_text = str(ds.getncattr(attribute_name))
    else:
        attribute_text = ""

    # Keep an optional leading ! so explicitly disabled options remain
    # distinguishable. Uppercase matching follows CPP macro conventions.
    tokens = {
        token.upper()
        for token in re.findall(
            r"!?[A-Za-z_][A-Za-z0-9_.]*",
            attribute_text,
        )
    }

    option_states = {}
    for option_name in option_names:
        normalized_name = option_name.upper()
        option_states[option_name] = (
            normalized_name in tokens
            and f"!{normalized_name}" not in tokens
        )

    return option_states


def get_cpp_parameter_values(ds):
    """Extract CPP states and their conditional scalar float values."""

    option_states = get_cpp_option_states(ds)
    results = dict(option_states)

    enabled_specs = {}
    for option_name in CPP_OPTION_PARAMETERS:
        variable_specs = CPP_CONDITIONAL_FLOAT_VARIABLES.get(
            option_name, {}
        )

        # Disabled options may legitimately omit their variables. Record a
        # consistent numerical baseline without trying to read them.
        if not option_states[option_name]:
            for output_names in variable_specs.values():
                for output_name in output_names:
                    results[output_name] = 0.0
        else:
            enabled_specs[option_name] = variable_specs

    if not enabled_specs:
        return results

    # Read enabled conditional variables through h5netcdf. The my_env
    # netCDF4 build can return a reused/corrupted buffer for some small
    # one-dimensional variables in these large station files.
    with xr.open_dataset(
        ds.filepath(),
        engine="h5netcdf",
        decode_cf=False,
    ) as cpp_ds:
        for option_name, variable_specs in enabled_specs.items():
            for variable_name, output_names in variable_specs.items():
                if variable_name not in cpp_ds.variables:
                    raise ValueError(
                        f"CPP option {option_name} is enabled, but "
                        f"{variable_name} is missing"
                    )

                values = np.array(
                    cpp_ds[variable_name].values,
                    dtype=float,
                    copy=True,
                ).reshape(-1)
                if values.size != len(output_names):
                    raise ValueError(
                        f"CPP option {option_name} is enabled, but "
                        f"{variable_name} has {values.size} value(s); "
                        f"expected {len(output_names)}"
                    )

                for output_name, value in zip(output_names, values):
                    if not np.isfinite(value):
                        raise ValueError(
                            f"{output_name} is non-finite while "
                            f"CPP option {option_name} is enabled"
                        )
                    results[output_name] = float(value)

    return results

## 2.5. Representative biological-tracer extraction

`nl_tnu2_tracer9` is read from the numeric `nl_tnu2` station-file variable. The paired categorical advection scheme is read from the `detritus:` line of the multiline `NLM_TADV` global attribute. Complete but unconfigured pairs are retained as missing and reported before trial import.

In [8]:
def get_biological_advection_scheme(
    ds,
    tracer_name=BIO_TRACER_ATTRIBUTE_NAME,
    attribute_name="NLM_TADV",
):
    """Return the configured category matching one tracer's H/V pair."""

    if attribute_name not in ds.ncattrs():
        return np.nan

    match = re.search(
        rf"^\s*{re.escape(tracer_name)}:\s+(\S+)\s+(\S+)",
        str(ds.getncattr(attribute_name)),
        flags=re.MULTILINE,
    )
    if match is None:
        return np.nan

    observed_pair = match.groups()
    for category, expected_pair in BIO_ADVECTION_SCHEMES.items():
        if observed_pair == expected_pair:
            return category

    return np.nan

<a id="read-runs"></a>
# 📥 3. Read completed runs

Cycle through all runs listed in `FILE_LIST`, open each cost summary to calculate nRMSE and total cost, and open each station file to collect parameters, configured CPP-option states, conditional CPP values, and decoded `dstart_year`. File and metric problems are recorded for review. Runs with unusable costs are filtered in Section 4, while historical import separately rejects rows missing required Optuna parameters.

In [9]:
# ============================================================
# PROCESS EACH MODEL RUN ONCE
#   - RMSE metrics
#   - Cost
#   - Parameters
# ============================================================

all_results = []
rmse_diagnostics = []

missing_cost_files = []
missing_station_files = []
fallback_run_date_runs = []

for n, (_, row) in enumerate(runs.iterrows(), start=1):

    run_name = row["Run Name"]
    cost_file = row["Cost File"]
    station_file = row["Station File"]

    cost_path = COST_DIR / cost_file
    station_path = COST_DIR / station_file

    print(f"\n[{n}/{len(runs)}] Processing {run_name}")
    print(f"  Cost file:    {cost_path.name}")
    print(f"  Station file: {station_path.name}")

    # Everything for this run goes into ONE dictionary
    run_results = {
        "Run Name": run_name,
        "Cost File": cost_file,
        "Station File": station_file
    }

    # ========================================================
    # COST FILE
    #   1. Calculate RMSE metrics
    #   2. Calculate total cost
    # ========================================================

    try:

        with xr.open_dataset(cost_path) as ds:

            # --------------------------
            # RMSE metrics
            # --------------------------
            for metric_name, (var_mod, var_obs) in metric_info.items():

                rmse = calculate_rmse_star(
                    ds,
                    metric_name,
                    var_mod,
                    var_obs,
                    run_name,
                    rmse_diagnostics,
                    MAX_MISMATCH_LOCATIONS,
                )

                run_results[metric_name] = rmse

            # --------------------------
            # Combined cost
            # --------------------------
            cost = calculate_cost(
                ds,
                cost_components,
                cost_weights
            )

            run_results["Cost"] = cost

        run_results["Cost Status"] = "Success"

    except Exception as e:

        print(f"  COST FILE ERROR: {e}")

        run_results["Cost Status"] = f"ERROR: {e}"

        # Make sure expected outputs exist
        for metric_name in metric_info:
            run_results.setdefault(metric_name, np.nan)

        run_results.setdefault("Cost", np.nan)

        missing_cost_files.append(cost_path.name)

    # ========================================================
    # STATION FILE
    #   Extract model parameters
    # ========================================================

    try:

        with Dataset(station_path, mode="r") as ds:

            # Determine CPP states first. Enabled conditional arrays are
            # copied before the general station-parameter read begins.
            run_results.update(get_cpp_parameter_values(ds))

            for parameter in parameter_names:

                # Option-dependent variables are handled below so disabled
                # CPP options never trigger an unnecessary variable read.
                if parameter in CPP_CONDITIONAL_STATION_VARIABLES:
                    continue

                parameter_values = get_parameter_values(
                    ds,
                    parameter
                )

                run_results.update(parameter_values)

            run_results[DSTART_YEAR_COLUMN] = get_dstart_year(ds)

            # Preserve the run date recorded in this station file's history.
            run_date, used_run_date_fallback = get_run_date(ds)
            run_results[RUN_DATE_COLUMN] = run_date
            if used_run_date_fallback:
                fallback_run_date_runs.append(run_name)

            # nl_tnu2 is not part of the biological parameter list, so read
            # tracer9 explicitly from the station-file variable.
            nl_tnu2_values = get_parameter_values(ds, "nl_tnu2")
            run_results[BIO_NL_TNU2_PARAMETER] = nl_tnu2_values.get(
                BIO_NL_TNU2_PARAMETER,
                np.nan,
            )

            # Use detritus (station-file tracer9) as the representative
            # horizontal/vertical biological advection pair.
            run_results[BIO_ADVECTION_PARAMETER] = (
                get_biological_advection_scheme(ds)
            )

        run_results["Parameter Status"] = "Success"

    except Exception as e:

        print(f"  STATION FILE ERROR: {e}")

        run_results["Parameter Status"] = f"ERROR: {e}"

        # Fill parameters with NaN if station file cannot be read
        for parameter in parameter_names:
            run_results.setdefault(parameter, np.nan)

        run_results.setdefault(DSTART_YEAR_COLUMN, np.nan)
        run_results.setdefault(RUN_DATE_COLUMN, RUN_DATE_FALLBACK)
        if run_name not in fallback_run_date_runs:
            fallback_run_date_runs.append(run_name)
        run_results.setdefault(BIO_NL_TNU2_PARAMETER, np.nan)
        run_results.setdefault(BIO_ADVECTION_PARAMETER, np.nan)
        for parameter in CPP_RECORDED_PARAMETERS:
            run_results.setdefault(parameter, np.nan)

        missing_station_files.append(station_path.name)

    # ========================================================
    # SAVE THIS RUN
    # ========================================================

    all_results.append(run_results)

    gc.collect()


# ============================================================
# CREATE FINAL DATAFRAME
# ============================================================

DF_opt = pd.DataFrame(all_results)

# Safety cleanup
DF_opt = DF_opt.dropna(
    axis=1,
    how="all"
)

if fallback_run_date_runs:
    print(
        f"Runs assigned fallback date {RUN_DATE_FALLBACK.date()}: "
        f"{fallback_run_date_runs}"
    )


[1/113] Processing A13
  Cost file:    A13.nc
  Station file: ocean_sta_A13.nc
  RMSE WARNING [Surface pH] missing-data mismatch: model-only=2559, observation-only=0
  RMSE WARNING [Bottom pH] missing-data mismatch: model-only=3078, observation-only=0
  RMSE WARNING [Surface Oxygen] missing-data mismatch: model-only=2559, observation-only=0
  RMSE WARNING [Bottom Oxygen] missing-data mismatch: model-only=3078, observation-only=0
  RMSE WARNING [Bottom Si] missing-data mismatch: model-only=6, observation-only=0
  RMSE WARNING [SOD] missing-data mismatch: model-only=31, observation-only=0

[2/113] Processing DU_Mar16
  Cost file:    DU_Mar16.nc
  Station file: ocean_sta_DU_Mar16.nc
  RMSE WARNING [Surface pH] missing-data mismatch: model-only=2559, observation-only=0
  RMSE WARNING [Bottom pH] missing-data mismatch: model-only=3078, observation-only=0
  RMSE WARNING [Surface Oxygen] missing-data mismatch: model-only=2559, observation-only=0
  RMSE WARNING [Bottom Oxygen] missing-data mi

## 3.1. Data-loading and RMSE diagnostics

This section reports failed input files, unusually large costs, missing metric values, and any model/observation pairing problems found while calculating RMSD*'. These cells summarize issues already recorded during the run-processing loop; they do not modify the data.

In [10]:
failed_cost_runs = DF_opt.loc[
    DF_opt["Cost Status"] != "Success", "Run Name"
].tolist()
failed_parameter_runs = DF_opt.loc[
    DF_opt["Parameter Status"] != "Success", "Run Name"
].tolist()
high_cost_runs = DF_opt.loc[
    DF_opt["Cost"] >= COST_THRESHOLD, "Run Name"
].tolist()

print(f"Runs with cost-file errors: {failed_cost_runs}")
print(f"Runs with station-file errors: {failed_parameter_runs}")
print(f"Runs at or above the cost threshold: {high_cost_runs}")

DF_rmse_diagnostics = pd.DataFrame(
    rmse_diagnostics,
    columns=["Run Name", "Metric", "Issue", "Details"],
)

if DF_rmse_diagnostics.empty:
    print("\nNo RMSE pairing or variance problems found.")
else:
    print(f"\nRMSE diagnostic records: {len(DF_rmse_diagnostics)}")
    display(DF_rmse_diagnostics)

Runs with cost-file errors: ['C13', 'OPTUNA_76', 'OPTUNA_77']
Runs with station-file errors: ['C13', 'OPTUNA_76', 'OPTUNA_77']
Runs at or above the cost threshold: ['D10', 'D16']

RMSE diagnostic records: 675


,Run Name,Metric,Issue,Details
0,A13,Surface pH,Missing-data mismatch,model finite/observation non-finite=2559 at ['...
1,A13,Bottom pH,Missing-data mismatch,model finite/observation non-finite=3078 at ['...
2,A13,Surface Oxygen,Missing-data mismatch,model finite/observation non-finite=2559 at ['...
3,A13,Bottom Oxygen,Missing-data mismatch,model finite/observation non-finite=3078 at ['...
4,A13,Bottom Si,Missing-data mismatch,model finite/observation non-finite=6 at ['Dat...
...,...,...,...,...
670,OPTUNA_75,Bottom pH,Missing-data mismatch,model finite/observation non-finite=3078 at ['...
671,OPTUNA_75,Surface Oxygen,Missing-data mismatch,model finite/observation non-finite=2559 at ['...
672,OPTUNA_75,Bottom Oxygen,Missing-data mismatch,model finite/observation non-finite=3078 at ['...
673,OPTUNA_75,Bottom Si,Missing-data mismatch,model finite/observation non-finite=6 at ['Dat...


In [11]:
print("Missing values per RMSE column:")
rmse_columns = list(metric_info)
print(DF_opt[rmse_columns].isna().sum())

print("\nAll values finite:")
print(np.isfinite(DF_opt[rmse_columns]).all())

Missing values per RMSE column:
Surface pH        3
Bottom pH         3
Surface Oxygen    3
Bottom Oxygen     3
Surface NO3       3
Bottom NO3        3
Surface NH4       3
Bottom NH4        3
Surface Si        4
Bottom Si         4
Secchi Depth      3
SOD               3
dtype: int64

All values finite:
Surface pH        False
Bottom pH         False
Surface Oxygen    False
Bottom Oxygen     False
Surface NO3       False
Bottom NO3        False
Surface NH4       False
Bottom NH4        False
Surface Si        False
Bottom Si         False
Secchi Depth      False
SOD               False
dtype: bool


<a id="prepare-data"></a>
# 🧹 4. Prepare the optimization dataframe

## 4.1. Objective nRMSE 
The signed nRMSE values remain available for interpretation. Separate objective columns use their absolute values so Optuna can minimize departure magnitude regardless of sign.

In [12]:
active_rmse_cols = list(metric_info)

active_objective_cols = []

for col in active_rmse_cols:
    new_col = f"{col} Objective"
    DF_opt[new_col] = DF_opt[col].abs()
    active_objective_cols.append(new_col)
    
DF_opt[active_objective_cols].describe().T[["min",
                                            "25%",
                                            "50%",
                                            "75%",
                                            "max"]]

,min,25%,50%,75%,max
Surface pH Objective,0.681433,0.841094,0.915853,1.098894,4.189569
Bottom pH Objective,0.641253,0.783798,0.966933,1.167659,4.927350
Surface Oxygen Objective,0.592770,0.665101,0.732195,0.956439,1.096477
Bottom Oxygen Objective,0.550607,0.598118,0.737895,0.821473,1.098110
Surface NO3 Objective,0.675214,0.860902,1.135514,1.402419,3.348916
Bottom NO3 Objective,0.679697,0.963341,1.242624,1.591019,3.022346
Surface NH4 Objective,0.777591,0.834414,0.875509,0.914694,1.544785
Bottom NH4 Objective,0.919232,1.009848,1.092540,1.221103,2.584304
Surface Si Objective,0.645206,0.723225,0.841976,0.896471,1.030878
Bottom Si Objective,0.733499,0.842114,0.927742,0.970083,1.290688


## 4.2. Keep runs with usable costs

Runs with missing or non-finite costs, along with runs at or above `COST_THRESHOLD`, are excluded from optimization.

In [13]:
print(
    f"Keeping runs with finite costs below {COST_THRESHOLD}. "
    "Missing and non-finite costs are also excluded.\n"
)

old_len = len(DF_opt)

usable_cost = (
    np.isfinite(DF_opt["Cost"])
    & (DF_opt["Cost"] < COST_THRESHOLD)
)
DF_opt = DF_opt.loc[usable_cost].reset_index(drop=True)

new_len = len(DF_opt)

print("Number of runs:", len(DF_opt))
print("Number of runs dropped:", old_len - new_len)

del old_len, new_len

Keeping runs with finite costs below 200. Missing and non-finite costs are also excluded.

Number of runs: 107
Number of runs dropped: 6


## 4.3. Identify varied numeric and categorical parameters

Numeric parameters are available for continuous diagnostics such as min/max summaries, correlations, and regressions. The representative advection scheme and configured Boolean CPP options are retained separately as categorical parameters so they are never passed to numeric calculations.

In [14]:
metadata_columns = {
    "Run Name", "Station File", "Parameter Status",
    "Cost File", "Cost", "Cost Status", RUN_DATE_COLUMN,
    *active_rmse_cols, *active_objective_cols,
}

candidate_parameter_cols = [
    column for column in DF_opt.columns
    if column not in metadata_columns
]

numeric_parameter_cols = [
    column for column in candidate_parameter_cols
    if (
        column not in CPP_OPTION_PARAMETERS
        and pd.api.types.is_numeric_dtype(DF_opt[column])
    )
]
categorical_parameter_cols = [
    column for column in candidate_parameter_cols
    if column not in numeric_parameter_cols
]

In [15]:
numeric_parameter_variation = (
    DF_opt[numeric_parameter_cols].nunique(dropna=True).sort_values()
)
categorical_parameter_variation = (
    DF_opt[categorical_parameter_cols].nunique(dropna=True).sort_values()
)

varied_parameter_cols = numeric_parameter_variation[
    numeric_parameter_variation > 1
].index.tolist()
varied_categorical_parameter_cols = categorical_parameter_variation[
    categorical_parameter_variation > 1
].index.tolist()
optimization_parameter_cols = (
    varied_parameter_cols + varied_categorical_parameter_cols
)

print("Varied numeric parameters:", len(varied_parameter_cols))
print(varied_parameter_cols)
print("\nVaried categorical parameters:", len(varied_categorical_parameter_cols))
print(varied_categorical_parameter_cols)

Varied numeric parameters: 63
['bUmaxSi_nspc0', 'bUmaxSi_nspc1', 'bpsi_n', 'bao2', 'o2nh', 'aksio4s2', 'c2n', 'bfp_nspc2', 'bfn_nspc0', 'bfp_nspc0', 'bfn_nspc1', 'bfn_nspc2', 'bfc_nspc0', 'bfc_nspc2', 'bfp_nspc1', 'bfs_nspc2', 'akpo4s1', 'pis2', 'pis1', 'cacodr', 'omega_thresh', 'wsPCa', 'ro5', 'o2no', 'ro6', 'aknh4s1', 'bgamma0', 'cacopf', 'Chl2cs1_m', 'bgamma5s', 'akno3s1', 'wsp', 'nl_tnu2_tracer9', 'bfs_nspc0', 'bfs_nspc1', 'bpsi_p', 'wsdsi', 'Chl2cs2_m', 'akno3s2', 'bgamma3', 'rrb1', 'rrb2', 'rrg2', 'rrg1', 'gmaxs1', 'amaxs1', 'bgamma4', 'gmaxs2', 'amaxs2', 'akz1', 'akz2', 'reg2', 'reg1', 'bgamma6', 'beta1', 'beta2', 'bUmax_nspc0', 'bnit', 'wsd', 'bUmax_nspc1', 'bdenit', 'bgamma7', 'bgamma5']

Varied categorical parameters: 2
['CACO3', 'tracer9_advection_scheme']


## 4.4. Build the Optuna dataframe

The optimization table retains the run name, total cost, parameters that vary, configured CPP-option parameters, and the absolute values of the signed RMSD*' metrics. CPP-option states and their conditional floats are retained even when they are constant across all historical runs. Total cost is the scalar Optuna objective; individual metrics are stored as trial metadata for diagnosis.

In [16]:
missing_cpp_columns = [
    parameter
    for parameter in CPP_RECORDED_PARAMETERS
    if parameter not in DF_opt.columns
]
if missing_cpp_columns:
    raise ValueError(
        f"Configured CPP parameters are unavailable: {missing_cpp_columns}"
    )

# dict.fromkeys preserves order while preventing duplicate columns if a
# recorded CPP field also varies among a future set of historical runs.
optuna_columns = list(dict.fromkeys([
    "Run Name",
    RUN_DATE_COLUMN,
    "Cost",
    *CPP_RECORDED_PARAMETERS,
    *optimization_parameter_cols,
    *active_objective_cols,
]))
DF_optuna = DF_opt[optuna_columns].copy()

print(DF_optuna.shape)
display(DF_optuna.head())

(107, 83)


,Run Name,Run Date,Cost,CACO3,cacopf,cacodr,omega_thresh,wsPCa,bUmaxCa_nspc0,bUmaxCa_nspc1,...,Surface Oxygen Objective,Bottom Oxygen Objective,Surface NO3 Objective,Bottom NO3 Objective,Surface NH4 Objective,Bottom NH4 Objective,Surface Si Objective,Bottom Si Objective,Secchi Depth Objective,SOD Objective
0,A13,2026-01-18,55.697850,False,0.00,0.0000,0.0,0.0,0.0,0.0,...,0.652691,0.580342,0.942058,1.107766,0.841999,0.970543,0.752158,0.851464,1.236557,1.066500
1,DU_Mar16,2026-03-12,56.506814,False,0.00,0.0000,0.0,0.0,0.0,0.0,...,1.054174,0.899141,1.236161,1.340496,0.838992,1.049942,0.969270,0.971109,1.089839,1.063369
2,DU,2026-03-07,53.119913,False,0.00,0.0000,0.0,0.0,0.0,0.0,...,1.096477,1.063434,1.135046,1.065471,0.800463,1.186002,0.991277,0.970083,1.058675,1.063521
3,A17,2026-01-22,55.447151,False,0.00,0.0000,0.0,0.0,0.0,0.0,...,0.652900,0.587009,0.924558,1.090406,0.824347,0.957371,0.747530,0.846235,1.240319,1.071099
4,C11,2026-04-23,49.140464,True,0.07,0.0057,1.0,2.0,0.0,0.0,...,0.834153,0.703119,0.807021,0.886873,0.899166,1.040742,0.983584,1.254989,1.129166,0.950199


<a id="optuna-study"></a>
# 🧠 5. Load the Optuna study

## 5.1. Load the study

Create the configured SQLite study when it does not exist, or load it when it does. The objective-version label is checked immediately afterward so incompatible cost definitions cannot be mixed.

[Watanabe (2023)](https://doi.org/10.48550/arXiv.2304.11127)

In [17]:
# ============================================================
# CREATE OR LOAD STUDY
# ============================================================

# >>> USER INPUT (advanced): Review sampler settings only when changing the
# exploration strategy or reproducibility seed.
sampler = optuna.samplers.TPESampler(
    n_startup_trials=10,
    multivariate=True,
    seed=42
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction="minimize",
    sampler=sampler,
    load_if_exists=True
)

print(f"\nLoaded study: {study.study_name}")
print(f"Number of trials currently stored: {len(study.trials)}")

/var/folders/9k/5r38tm8d21g6nm3w9rchrd6m0000gn/T/ipykernel_86524/1333633658.py:7: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-09-19 13:08:05,697] A new study created in RDB with name: model_calibration_bio



Loaded study: model_calibration_bio
Number of trials currently stored: 0


In [18]:
# ============================================================
# VERIFY STUDY OBJECTIVE VERSION
# ============================================================

stored_objective_version = study.user_attrs.get(
    "Objective Version"
)

if stored_objective_version is None:

    study.set_user_attr(
        "Objective Version",
        OBJECTIVE_VERSION
    )

    print(
        f"Assigned objective version: "
        f"{OBJECTIVE_VERSION}"
    )

elif stored_objective_version != OBJECTIVE_VERSION:

    raise ValueError(
        "Objective-version mismatch:\n"
        f"  Study:    {stored_objective_version}\n"
        f"  Notebook: {OBJECTIVE_VERSION}\n\n"
        "Do not combine results calculated with different "
        "objective definitions in the same Optuna study."
    )

else:

    print(
        f"Objective version confirmed: "
        f"{OBJECTIVE_VERSION}"
    )

Assigned objective version: cost_v3_bio_no_temp_salt


## 5.2. Parameter distributions

Numeric ranges default to observed minima and maxima, with the existing manual `bdenit` override. The representative tracer-9 advection pair uses an explicit categorical distribution. Recorded CPP options use Boolean categorical distributions, while their conditional floats use observed ranges that may be fixed when all historical values are `0.0`. Recorded CPP fields are included in historical trials but are not sampled when new candidates are generated. Every parameter must have a valid distribution before the study is modified.

<div class="alert alert-info"><strong>USER INPUT — active parameters:</strong> Edit <code>active_numeric_parameters</code> and <code>active_categorical_parameters</code> in the next code cell. Valid numeric names and their bounds are displayed before the active selection. Valid categorical names and choices are printed separately. Do not place a categorical parameter in the numeric list or vice versa.</div>

In [19]:
# ============================================================
# NUMERIC PARAMETER LOOKUP TABLE
# ============================================================

parameter_summary = pd.DataFrame({
    "Parameter": varied_parameter_cols,
    "Low": [DF_optuna[parameter].min() for parameter in varied_parameter_cols],
    "High": [DF_optuna[parameter].max() for parameter in varied_parameter_cols],
    "Unique Values": [
        DF_optuna[parameter].nunique(dropna=True)
        for parameter in varied_parameter_cols
    ],
})
parameter_summary = parameter_summary.loc[
    parameter_summary["Unique Values"] > 1
].reset_index(drop=True)

all_parameter_bounds = {
    row["Parameter"]: (float(row["Low"]), float(row["High"]))
    for _, row in parameter_summary.iterrows()
}

# >>> USER INPUT: Scientific bound override for bdenit. Add other manual
# bounds here when observed historical extrema are not appropriate.
if "bdenit" not in DF_optuna.columns:
    raise ValueError("Cannot override bdenit bounds: parameter is unavailable.")
all_parameter_bounds["bdenit"] = (1.0, float(DF_optuna["bdenit"].max()))

# Keep only finite, increasing ranges. Constant or non-finite numeric fields
# are not valid continuous search choices.
available_numeric_parameter_bounds = {
    parameter: bounds
    for parameter, bounds in all_parameter_bounds.items()
    if np.isfinite(bounds).all() and bounds[0] < bounds[1]
}

# Display every valid numeric name and the bounds used when selected below.
available_numeric_parameter_table = pd.DataFrame([
    {
        "Parameter": parameter,
        "Low": low,
        "High": high,
    }
    for parameter, (low, high) in (
        available_numeric_parameter_bounds.items()
    )
]).sort_values("Parameter").reset_index(drop=True)

available_categorical_choices = {
    BIO_ADVECTION_PARAMETER: tuple(BIO_ADVECTION_SCHEMES),
}

print("Valid numeric parameter names and bounds:")
display(available_numeric_parameter_table)

print("\nValid categorical parameter names and choices:")
for parameter, choices in available_categorical_choices.items():
    print(f"  {parameter}: {choices}")

# ============================================================
# USER INPUT 11: PARAMETERS TO OPTIMIZE
# ============================================================

# >>> USER INPUT 11A: Select numeric parameter names only from the table
# displayed above. These are suggested as continuous floating-point values.
active_numeric_parameters = [
    "bgamma7", "bgamma5", "bUmax_nspc1", "wsd", "bnit", "bdenit",
    "beta1", "beta2", "bgamma6", "bgamma3", "bgamma4", "reg1",
    "reg2", "rrb1", "rrb2", "rrg1", "rrg2",
    BIO_NL_TNU2_PARAMETER,
]

# >>> USER INPUT 11B: Select categorical names only from the choices printed
# above. Optuna will select one of the listed choices for each parameter.
active_categorical_parameters = [BIO_ADVECTION_PARAMETER]
active_parameters = active_numeric_parameters + active_categorical_parameters

duplicate_active_parameters = [
    parameter
    for parameter in set(active_parameters)
    if active_parameters.count(parameter) > 1
]
if duplicate_active_parameters:
    raise ValueError(
        f"Active parameters are duplicated: {duplicate_active_parameters}"
    )

missing_numeric = [
    parameter for parameter in active_numeric_parameters
    if parameter not in available_numeric_parameter_bounds
]
if missing_numeric:
    raise ValueError(
        "Active numeric parameters lack usable historical bounds: "
        f"{missing_numeric}"
    )

missing_categorical = [
    parameter for parameter in active_categorical_parameters
    if parameter not in available_categorical_choices
]
if missing_categorical:
    raise ValueError(
        "Active categorical parameters lack configured choices: "
        f"{missing_categorical}"
    )

parameter_bounds = {
    parameter: available_numeric_parameter_bounds[parameter]
    for parameter in active_numeric_parameters
}
invalid_bounds = {
    parameter: bounds
    for parameter, bounds in parameter_bounds.items()
    if not np.isfinite(bounds).all() or bounds[0] >= bounds[1]
}
if invalid_bounds:
    raise ValueError(f"Invalid Optuna parameter bounds: {invalid_bounds}")

categorical_choices = {
    parameter: available_categorical_choices[parameter]
    for parameter in active_categorical_parameters
}

# ============================================================
# CPP PARAMETERS RECORDED FOR HISTORICAL TRIALS
# ============================================================

# Equal low/high values are valid here: all current CACO3-off runs use
# 0.0, and these fields describe historical runs rather than search bounds.
recorded_cpp_float_bounds = {
    parameter: (
        float(DF_optuna[parameter].min()),
        float(DF_optuna[parameter].max()),
    )
    for parameter in CPP_RECORDED_FLOAT_PARAMETERS
}
invalid_recorded_cpp_bounds = {
    parameter: bounds
    for parameter, bounds in recorded_cpp_float_bounds.items()
    if not np.isfinite(bounds).all() or bounds[0] > bounds[1]
}
if invalid_recorded_cpp_bounds:
    raise ValueError(
        "Invalid recorded CPP parameter bounds: "
        f"{invalid_recorded_cpp_bounds}"
    )

recorded_cpp_categorical_choices = {
    parameter: (False, True)
    for parameter in CPP_OPTION_PARAMETERS
}
historical_trial_parameters = [
    *active_parameters,
    *CPP_RECORDED_PARAMETERS,
]

print("Numeric parameters being optimized:")
for parameter, (low, high) in parameter_bounds.items():
    print(f"  {parameter}: {low} - {high}")

print("\nCategorical parameters being optimized:")
for parameter, choices in categorical_choices.items():
    print(f"  {parameter}: {choices}")

print("\nCPP parameters recorded for historical trials:")
for parameter in CPP_OPTION_PARAMETERS:
    print(f"  {parameter}: categorical {recorded_cpp_categorical_choices[parameter]}")
for parameter, (low, high) in recorded_cpp_float_bounds.items():
    print(f"  {parameter}: float {low} - {high}")

Valid numeric parameter names and bounds:


,Parameter,Low,High
0,Chl2cs1_m,0.02000,0.04
1,Chl2cs2_m,0.02675,0.06
2,aknh4s1,0.01000,0.10
3,akno3s1,0.75000,1.50
4,akno3s2,1.00000,3.00
...,...,...,...
58,rrg2,0.05000,0.30
59,wsPCa,0.00000,2.00
60,wsd,1.00000,5.00
61,wsdsi,1.00000,5.00



Valid categorical parameter names and choices:
  tracer9_advection_scheme: ('Upstream3_Centered4', 'HSIMT', 'Akima4', 'MPDATA')
Numeric parameters being optimized:
  bgamma7: 0.05 - 0.5
  bgamma5: 0.05 - 2.0
  bUmax_nspc1: 0.00274 - 0.0274
  wsd: 1.0 - 5.0
  bnit: 0.15 - 0.2987851871698918
  bdenit: 1.0 - 28.979636342517402
  beta1: 0.5 - 2.25
  beta2: 0.15 - 2.0
  bgamma6: 0.005 - 0.1
  bgamma3: 0.1 - 0.3
  bgamma4: 0.05 - 0.3
  reg1: 0.1 - 0.4
  reg2: 0.05 - 0.2
  rrb1: 0.1 - 0.4
  rrb2: 0.1 - 0.4
  rrg1: 0.05 - 0.3
  rrg2: 0.05 - 0.3
  nl_tnu2_tracer9: 0.5 - 4.0

Categorical parameters being optimized:
  tracer9_advection_scheme: ('Upstream3_Centered4', 'HSIMT', 'Akima4', 'MPDATA')

CPP parameters recorded for historical trials:
  CACO3: categorical (False, True)
  cacopf: float 0.0 - 0.14
  cacodr: float 0.0 - 0.0057
  omega_thresh: float 0.0 - 1.0
  wsPCa: float 0.0 - 2.0
  bUmaxCa_nspc0: float 0.0 - 0.0
  bUmaxCa_nspc1: float 0.0 - 0.0
  bUmaxCa_nspc2: float 0.0 - 0.0


## 5.3. Add historical runs

Each eligible dataframe row becomes one completed Optuna trial. Existing run names are skipped, and validation prevents missing parameters, invalid costs, out-of-bound values, or unknown categories from entering the study. A fresh database receives historical runs in stable run-date order. This historical import occurs regardless of the four action switches.

<div class="alert alert-danger"><strong>Database write:</strong> This section always adds eligible historical runs that are not already represented by a matching <code>Run Name</code>. The four action switches do not disable historical import.</div>

| Old Run Name | New Run Name | Station File |
| :--- | :--- | :--- |
| 2005_run1z_zi_modphys2_chlor2 | Dave1 | ocean_sta_Dave1_2005.nc |
| 2005_run1z_zi_modphys2_chlor2_chl2c | Dave2 | ocean_sta_Dave2_2005.nc |
| 2005_run1z_zj | Dave3 | ocean_sta_Dave3_2005.nc |
| 2005_run1z_zk | Dave4 | ocean_sta_Dave4_2005.nc |
| 2005_run1z_zl | Dave5 | ocean_sta_Dave5_2005.nc |
| 2005_run1z_zl_impl2 | Dave6 | ocean_sta_Dave6_2005.nc |

In [20]:
# ============================================================
# ADD HISTORICAL COMPLETED RUNS TO OPTUNA
# ============================================================

existing_run_names = {
    trial.user_attrs.get("Run Name")
    for trial in study.trials
    if trial.user_attrs.get("Run Name") is not None
}

float_distributions = {
    parameter: optuna.distributions.FloatDistribution(low=low, high=high)
    for parameter, (low, high) in parameter_bounds.items()
}
category_distributions = {
    parameter: optuna.distributions.CategoricalDistribution(choices=choices)
    for parameter, choices in categorical_choices.items()
}
recorded_cpp_float_distributions = {
    parameter: optuna.distributions.FloatDistribution(low=low, high=high)
    for parameter, (low, high) in recorded_cpp_float_bounds.items()
}
recorded_cpp_category_distributions = {
    parameter: optuna.distributions.CategoricalDistribution(choices=choices)
    for parameter, choices in recorded_cpp_categorical_choices.items()
}
trial_distributions = {
    **float_distributions,
    **category_distributions,
    **recorded_cpp_float_distributions,
    **recorded_cpp_category_distributions,
}

counts = {
    "Added": 0,
    "Already present": 0,
    "Missing parameters": 0,
    "Missing Cost": 0,
    "Outside bounds": 0,
    "Unknown category": 0,
}

# Stable sorting makes fresh-study trial numbers follow model run dates.
# Runs assigned the same date retain their order from DF_optuna.
historical_trials = DF_optuna.sort_values(
    RUN_DATE_COLUMN,
    kind="stable",
)

for _, row in historical_trials.iterrows():
    run_name = row["Run Name"]

    if run_name in existing_run_names:
        counts["Already present"] += 1
        continue
    if row[historical_trial_parameters].isna().any():
        counts["Missing parameters"] += 1
        continue
    if pd.isna(row["Cost"]) or not np.isfinite(row["Cost"]):
        counts["Missing Cost"] += 1
        continue
    if any(
        not (low <= float(row[parameter]) <= high)
        for parameter, (low, high) in parameter_bounds.items()
    ):
        counts["Outside bounds"] += 1
        continue
    if any(
        not (low <= float(row[parameter]) <= high)
        for parameter, (low, high) in recorded_cpp_float_bounds.items()
    ):
        counts["Outside bounds"] += 1
        continue
    if any(
        row[parameter] not in choices
        for parameter, choices in categorical_choices.items()
    ):
        counts["Unknown category"] += 1
        continue
    if any(
        row[parameter] not in choices
        for parameter, choices in recorded_cpp_categorical_choices.items()
    ):
        counts["Unknown category"] += 1
        continue

    params = {
        parameter: float(row[parameter])
        for parameter in active_numeric_parameters
    }
    params.update({
        parameter: str(row[parameter])
        for parameter in active_categorical_parameters
    })
    params.update({
        parameter: float(row[parameter])
        for parameter in CPP_RECORDED_FLOAT_PARAMETERS
    })
    params.update({
        parameter: bool(row[parameter])
        for parameter in CPP_OPTION_PARAMETERS
    })

    objective_values = {
        objective: (
            float(row[objective])
            if pd.notna(row[objective]) and np.isfinite(row[objective])
            else None
        )
        for objective in active_objective_cols
    }

    completed_trial = optuna.trial.create_trial(
        params=params,
        distributions=trial_distributions,
        value=float(row["Cost"]),
        state=optuna.trial.TrialState.COMPLETE,
        user_attrs={
            "Run Name": run_name,
            "Source": "Historical",
            "Objective Version": OBJECTIVE_VERSION,
            "Objectives": objective_values,
            RUN_DATE_COLUMN: row[RUN_DATE_COLUMN].date().isoformat(),
        },
    )
    study.add_trial(completed_trial)
    existing_run_names.add(run_name)
    counts["Added"] += 1

print("Historical trial import:")
for label, count in counts.items():
    print(f"  {label}: {count}")
print(f"\nTotal Optuna trials: {len(study.trials)}")

Historical trial import:
  Added: 106
  Already present: 0
  Missing parameters: 1
  Missing Cost: 0
  Outside bounds: 0
  Unknown category: 0

Total Optuna trials: 106


<a id="candidates"></a>
# 🚀 6. Synchronize results and manage candidates

First preview or register results for existing `RUNNING` trials. Only afterward may Optuna create a new batch, preventing completed-but-unregistered runs from unnecessarily blocking candidate generation.

## 6.1. Register newly completed model runs

A stored `RUNNING` trial is eligible for completion when its mapped actual run name appears in the cleaned results table with a finite cost. If the trial is absent from the current study's run map, its existing run name is used. Mapped study run names and trial numbers must agree. Eligible trials are always listed. The Optuna database is modified only when `UPDATE_OPTUNA_WITH_COMPLETED_RUNS` is `True`. This synchronization occurs before new candidates are requested so completed trials do not unnecessarily block the next batch.

<div class="alert alert-danger"><strong>Conditional database write:</strong> When <code>UPDATE_OPTUNA_WITH_COMPLETED_RUNS=True</code>, eligible trials are permanently changed from <code>RUNNING</code> to <code>COMPLETE</code>. When it is <code>False</code>, this section is preview-only.</div>

In [21]:
# ============================================================
# FIND AND OPTIONALLY COMPLETE FINISHED RUNNING TRIALS
# ============================================================

duplicate_mask = DF_optuna["Run Name"].duplicated(keep=False)
duplicate_names = DF_optuna.loc[duplicate_mask, "Run Name"].unique()

conflicting_duplicate_names = []
for run_name in duplicate_names:
    duplicate_rows = DF_optuna.loc[
        DF_optuna["Run Name"] == run_name
    ]
    if len(duplicate_rows.drop_duplicates()) > 1:
        conflicting_duplicate_names.append(run_name)

if conflicting_duplicate_names:
    raise ValueError(
        "Duplicate run names have conflicting optimization values: "
        f"{conflicting_duplicate_names}"
    )

if len(duplicate_names) > 0:
    print(
        "Removing exact duplicate optimization rows for: "
        f"{duplicate_names.tolist()}"
    )
    DF_optuna = DF_optuna.drop_duplicates(
        subset=["Run Name"],
        keep="first",
    ).reset_index(drop=True)

results_by_run = DF_optuna.set_index("Run Name", drop=False)
run_map_by_study_run_name = current_study_run_map.set_index(
    "Study Run Name", drop=False
)
run_map_by_trial_number = current_study_run_map.set_index(
    "Trial Number", drop=False
)
running_before_update = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]

completion_rows = []
n_completed_now = 0
n_results_not_available = 0
n_missing_run_name = 0

for frozen_trial in running_before_update:
    study_run_name = frozen_trial.user_attrs.get("Run Name")

    if not study_run_name:
        n_missing_run_name += 1
        print(f"Trial {frozen_trial.number}: missing Run Name attribute")
        continue

    if study_run_name in run_map_by_study_run_name.index:
        mapping = run_map_by_study_run_name.loc[study_run_name]
        mapped_trial_number = int(mapping["Trial Number"])
        if mapped_trial_number != frozen_trial.number:
            raise ValueError(
                f"Run-map mismatch for study {CURRENT_STUDY_NAME!r}: "
                f"Study Run Name {study_run_name!r} maps to trial "
                f"{mapped_trial_number}, but the RUNNING Optuna trial is "
                f"{frozen_trial.number}."
            )
        actual_run_name = mapping["Actual Run Name"]
    elif frozen_trial.number in run_map_by_trial_number.index:
        mapping = run_map_by_trial_number.loc[frozen_trial.number]
        raise ValueError(
            f"Run-map mismatch for study {CURRENT_STUDY_NAME!r}: trial "
            f"{frozen_trial.number} maps to Study Run Name "
            f"{mapping['Study Run Name']!r}, but the trial stores "
            f"{study_run_name!r}."
        )
    else:
        # The run map is an override table; unmapped trials keep existing behavior.
        actual_run_name = study_run_name

    if actual_run_name not in results_by_run.index:
        n_results_not_available += 1
        continue

    row = results_by_run.loc[actual_run_name]
    cost = float(row["Cost"])
    objective_values = {
        objective: (
            float(row[objective])
            if pd.notna(row[objective]) and np.isfinite(row[objective])
            else None
        )
        for objective in active_objective_cols
    }

    completion_rows.append({
        "Study Name": CURRENT_STUDY_NAME,
        "Actual Run Name": actual_run_name,
        "Study Run Name": study_run_name,
        "Trial Number": frozen_trial.number,
        RUN_DATE_COLUMN: row[RUN_DATE_COLUMN],
        "Cost": cost,
        "Action": (
            "Complete in Optuna"
            if UPDATE_OPTUNA_WITH_COMPLETED_RUNS
            else "Preview only"
        ),
    })

    if not UPDATE_OPTUNA_WITH_COMPLETED_RUNS:
        continue

    # Recreate a live Trial solely to attach metadata before study.tell().
    live_trial = optuna.trial.Trial(study, frozen_trial._trial_id)
    live_trial.set_user_attr("Study Run Name", study_run_name)
    live_trial.set_user_attr("Actual Run Name", actual_run_name)
    live_trial.set_user_attr("Objectives", objective_values)
    live_trial.set_user_attr(
        RUN_DATE_COLUMN, row[RUN_DATE_COLUMN].date().isoformat()
    )
    study.tell(frozen_trial.number, cost)
    n_completed_now += 1

DF_completion_candidates = pd.DataFrame(
    completion_rows,
    columns=[
        "Study Name", "Actual Run Name", "Study Run Name",
        "Trial Number", RUN_DATE_COLUMN, "Cost", "Action",
    ],
)

if DF_completion_candidates.empty:
    print("No RUNNING trials have completed results available.")
else:
    display(DF_completion_candidates)

print("\nOptuna completion summary:")
print(f"  Update enabled:          {UPDATE_OPTUNA_WITH_COMPLETED_RUNS}")
print(f"  Eligible for completion: {len(completion_rows)}")
print(f"  Completed now:           {n_completed_now}")
print(f"  Results not available:   {n_results_not_available}")
print(f"  Missing Run Name:        {n_missing_run_name}")

No RUNNING trials have completed results available.

Optuna completion summary:
  Update enabled:          False
  Eligible for completion: 0
  Completed now:           0
  Results not available:   0
  Missing Run Name:        0


In [22]:
# ============================================================
# REPORT STUDY STATUS AND LOAD CURRENT RUNNING CANDIDATES
# ============================================================

print(f"Study: {study.study_name}")
print(f"Total trials: {len(study.trials)}")

for state in optuna.trial.TrialState:
    count = sum(trial.state == state for trial in study.trials)
    if count > 0:
        print(f"  {state.name}: {count}")

running_trials = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]

print(f"Running trials: {len(running_trials)}")
candidate_results = []

for trial in running_trials:
    
    print(
        f"  Trial {trial.number}: "
        f"{trial.user_attrs.get('Run Name', 'No Run Name')}"
    )

    candidate_results.append({
        "Run Name": trial.user_attrs.get("Run Name"),
        "Trial Number": trial.number,
        **trial.params
    })

DF_candidates = pd.DataFrame(
    candidate_results,
    columns=["Run Name", "Trial Number", *active_parameters],
)

display(DF_candidates)

Study: model_calibration_bio
Total trials: 106
  COMPLETE: 106
Running trials: 0


,Run Name,Trial Number,bgamma7,bgamma5,bUmax_nspc1,wsd,bnit,bdenit,beta1,beta2,...,bgamma3,bgamma4,reg1,reg2,rrb1,rrb2,rrg1,rrg2,nl_tnu2_tracer9,tracer9_advection_scheme


## 6.2. Generate candidate parameter sets

New trials are requested only when `GENERATE_NEW_CANDIDATES` is `True` and no trials remain `RUNNING`. The setting itself is never changed by the notebook. Existing running candidates remain available for input-file regeneration.

<div class="alert alert-danger"><strong>Conditional database write:</strong> When <code>GENERATE_NEW_CANDIDATES=True</code>, Optuna creates new <code>RUNNING</code> trials and the notebook writes a candidate-summary CSV. Leave it <code>False</code> for review-only runs.</div>

In [23]:
# ============================================================
# GENERATE NEW OPTUNA CANDIDATES
# ============================================================

running_trials = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]
generated_new_candidates = False

if GENERATE_NEW_CANDIDATES and running_trials:
    print(
        f"Not generating candidates because {len(running_trials)} "
        "trial(s) are already RUNNING."
    )

elif GENERATE_NEW_CANDIDATES:
    candidate_results = []

    for _ in range(N_CANDIDATES):
        trial = study.ask()
        run_name = f"OPTUNA_BIO_{trial.number}"
        trial.set_user_attr("Run Name", run_name)
        trial.set_user_attr("Source", "Optuna suggestion")
        trial.set_user_attr("Objective Version", OBJECTIVE_VERSION)

        params = {
            parameter: trial.suggest_float(parameter, low, high)
            for parameter, (low, high) in parameter_bounds.items()
        }
        for parameter, choices in categorical_choices.items():
            params[parameter] = trial.suggest_categorical(parameter, choices)

        candidate_results.append({
            "Run Name": run_name,
            "Trial Number": trial.number,
            **params,
        })

    DF_candidates = pd.DataFrame(candidate_results)
    generated_new_candidates = True
    display(DF_candidates)

else:
    print("Candidate generation is OFF.")
    if running_trials:
        print(f"{len(running_trials)} trial(s) remain RUNNING.")

Candidate generation is OFF.


In [24]:
# ============================================================
# SAVE CANDIDATE PARAMETER SETS
# ============================================================

if generated_new_candidates:

    # >>> USER INPUT (optional): Change the candidate-summary filename if
    # multiple generated batches should be retained separately.
    candidate_file = OPT_DIR / "Optuna_Bio_Candidate_Parameters.csv"

    DF_candidates.to_csv(
        candidate_file,
        index=False
    )

    print("Saved candidate parameters to:")
    print(candidate_file)

<a id="candidate-files"></a>
# 📝 7. Create candidate input files

This section audits where every candidate value comes from, generates both input-file texts in memory, validates replacement counts, and writes files only when explicitly enabled. Original templates are read but never modified.

<div class="alert alert-warning">
<h4>Limitation: CPP options and candidate input files</h4>
<p>CPP-option states extracted from historical station files are recorded as categorical Optuna parameters for analysis. They are not currently used to modify generated ROMS input files, select an executable, or change compile-time CPP definitions.</p>
<p>In particular, recording <code>CACO3</code> does not enable or disable CACO3 for a newly generated candidate. Candidate generation and input-file writing must therefore use an executable and templates whose CPP configuration is managed separately. Support for proposing CPP-option states and writing compatible candidate files can be added once that workflow is defined.</p>
</div>

## 7.1. Files and templates

Templates are read for parameter auditing even when file writing is disabled. `OUTPUT_DIR` is created only when `WRITE_CANDIDATE_INPUT_FILES` is `True` and at least one running candidate is available. Replacement helpers require exactly one matching active block so incompatible templates fail before any file is written.

In [ ]:
# ============================================================
# USER INPUT 12: TEMPLATE INPUT FILES
# ============================================================

# >>> USER INPUT: Confirm both templates match the executable and model
# configuration used for the next candidate batch.
OPTICS_TEMPLATE = Path(
    "/Users/akbaskind/Desktop/LHS/"
    "bio_UMAINE15_sediment_optics_LHS1.in"
)

SEDIMENT_TEMPLATE = Path(
    "/Users/akbaskind/Desktop/LHS/"
    "bio_UMAINE15_sedmodel_LHS1.in"
)


# ============================================================
# USER INPUT 13: CANDIDATE OUTPUT DIRECTORY
# ============================================================

# >>> USER INPUT: Generated candidate files are written here only when the
# WRITE_CANDIDATE_INPUT_FILES switch is True.
OUTPUT_DIR = OPT_DIR / "Candidate_Input_Files"


def format_roms_value(value):
    """Format a numeric ROMS input value with D exponent notation."""

    text = f"{float(value):.16g}"

    if "." not in text and "e" not in text.lower():
        text += ".0"

    return f"{text}d0"


def replace_biological_tnu2(text, value):
    """Apply one nl_tnu2 value to all biological tracers."""

    if not np.isfinite(value):
        raise ValueError(f"nl_tnu2 has non-finite value {value}")

    pattern = (
        rf"^(\s*TNU2\s*==\s*{BIOLOGICAL_TRACER_COUNT}\s*\*\s*)"
        rf"([^\s!]+)"
    )
    updated, count = re.subn(
        pattern,
        rf"\g<1>{format_roms_value(value)}",
        text,
        count=1,
        flags=re.MULTILINE,
    )
    if count != 1:
        raise ValueError(f"TNU2 had {count} active-line replacements")
    return updated


def replace_advection_block(text, keyword, scheme_code, expected_values=15):
    """Replace every value in one Hadvection or Vadvection block."""

    lines = text.splitlines(keepends=True)
    start_indices = [
        index for index, line in enumerate(lines)
        if re.match(rf"^\s*{re.escape(keyword)}\s*==", line)
    ]
    if len(start_indices) != 1:
        raise ValueError(f"{keyword} had {len(start_indices)} active blocks")

    start = start_indices[0]
    for offset in range(expected_values):
        index = start + offset
        if index >= len(lines):
            raise ValueError(f"{keyword} ended before value {offset + 1}")

        if offset == 0:
            pattern = rf"^(\s*{re.escape(keyword)}\s*==\s*)(\S+)(.*)$"
        else:
            pattern = r"^(\s*)(\S+)(\s+.*)$"

        updated, count = re.subn(
            pattern,
            rf"\g<1>{scheme_code}\g<3>",
            lines[index],
            count=1,
        )
        if count != 1 or f"idbio({offset + 1:2d})" not in updated:
            raise ValueError(
                f"{keyword} value {offset + 1} did not match the expected template line"
            )
        lines[index] = updated

    return "".join(lines)


## 7.2. Input-file parameter inventories

These lists identify which model parameters belong to the water-column/optics template and which belong to the sediment template. They drive baseline lookup, source auditing, and text replacement.

In [ ]:
# ============================================================
# USER INPUT 14: PARAMETERS PRESENT IN EACH INPUT FILE
# ============================================================

# >>> USER INPUT (advanced): Update these inventories only when the template
# structure or model parameter set changes.
optics_parameters = [
    "reg1",
    "reg2",
    "gmaxs1",
    "gmaxs2",
    "rrb1",
    "rrb2",
    "rrg1",
    "rrg2",
    "beta1",
    "beta2",
    "akz1",
    "akz2",
    "PARfrac",
    "amaxs1",
    "amaxs2",
    "parsats1",
    "parsats2",
    "pis1",
    "pis2",
    "akno3s1",
    "akno3s2",
    "aknh4s1",
    "aknh4s2",
    "akpo4s1",
    "akpo4s2",
    "akco2s1",
    "akco2s2",
    "aksio4s2",
    "ak1",
    "ak2",
    "bgamma0",
    "bgamma1",
    "bgamma2",
    "bgamma3",
    "bgamma4",
    "bgamma5",
    "bgamma5s",
    "bgamma6",
    "bgamma7",
    "wsd",
    "wsdsi",
    "wsp",
    "pco2a",
    "si2n",
    "p2n",
    "o2no",
    "o2nh",
    "c2n",
    "ro5",
    "ro6",
    "ro7",
    "akox",
    "Chl2cs1_m",
    "Chl2cs2_m",
]

sediment_parameters = [
    "bUmax_nspc0",
    "bUmax_nspc1",
    "bUmax_nspc2",
    "bUmaxSi_nspc0",
    "bUmaxSi_nspc1",
    "bUmaxSi_nspc2",
    "bdep",
    "balpha",
    "bw",
    "btheta_diag",
    "bnit",
    "btheta_nit",
    "bdo_nit",
    "bdenit",
    "btheta_denit",
    "bpsi_n",
    "bdo_c",
    "bao2",
    "bpi",
    "bpsi_p",
    "bfc_nspc0",
    "bfc_nspc1",
    "bfc_nspc2",
    "bfn_nspc0",
    "bfn_nspc1",
    "bfn_nspc2",
    "bfp_nspc0",
    "bfp_nspc1",
    "bfp_nspc2",
    "bfs_nspc0",
    "bfs_nspc1",
    "bfs_nspc2",
]

## 7.3. Active parameters

These are the active Optuna parameters assigned to each template. The preparation flag is separate from candidate generation, so files can be recreated for trials that are already `RUNNING`.

`nl_tnu2_tracer9` and `tracer9_advection_scheme` are special water-column controls. They are handled by validated block replacements rather than the scalar biological-parameter replacement loop.


In [ ]:
active_optics_parameters = [
    p for p in active_parameters
    if p in optics_parameters
]

active_sediment_parameters = [
    p for p in active_parameters
    if p in sediment_parameters
]

special_optics_parameters = [
    BIO_NL_TNU2_PARAMETER,
    BIO_ADVECTION_PARAMETER,
]

print("Active optics parameters:")
print(active_optics_parameters)

print("\nActive sediment parameters:")
print(active_sediment_parameters)

prepare_candidate_input_files = (
    WRITE_CANDIDATE_INPUT_FILES
    and not DF_candidates.empty
)

if WRITE_CANDIDATE_INPUT_FILES and DF_candidates.empty:
    print("No candidate input files will be written: no RUNNING candidates.")

print("\nSpecial optics parameters:")
print(special_optics_parameters)


## 7.4. Set inactive parameters from the baseline run

Parameters that Optuna is not varying inherit values from `BASELINE_RUN_NAME`. Values in `SCIENTIFIC_OVERRIDES` take precedence. The source audit below makes every resulting choice visible before candidate text is generated.

In [ ]:
# ============================================================
# BUILD FIXED CANDIDATE BASELINE
# ============================================================

baseline_matches = DF_opt.loc[
    DF_opt["Run Name"] == BASELINE_RUN_NAME
]

if len(baseline_matches) == 0:
    raise ValueError(
        f"Baseline run {BASELINE_RUN_NAME!r} was not found."
    )

if len(baseline_matches) > 1:
    raise ValueError(
        f"Multiple rows were found for {BASELINE_RUN_NAME!r}."
    )

baseline_run = baseline_matches.iloc[0]

# All parameters represented in the two candidate input files
input_file_parameters = (
    optics_parameters
    + sediment_parameters
)

# Parameters that baseline can supply
baseline_values = {
    parameter: float(baseline_run[parameter])
    for parameter in input_file_parameters
    if (
        parameter in DF_opt.columns
        and pd.notna(baseline_run[parameter])
    )
}

# Scientific overrides must remain fixed, not varied by Optuna
override_conflicts = (
    set(SCIENTIFIC_OVERRIDES)
    & set(active_parameters)
)

if override_conflicts:
    raise ValueError(
        "These parameters are both scientific overrides "
        "and active Optuna parameters: "
        f"{sorted(override_conflicts)}"
    )

# Start with Dave5, but exclude parameters Optuna will vary
fixed_candidate_values = {
    parameter: value
    for parameter, value in baseline_values.items()
    if parameter not in active_parameters
}

# Scientific decisions take precedence over Dave5
fixed_candidate_values.update(
    SCIENTIFIC_OVERRIDES
)

print(f"Baseline run: {BASELINE_RUN_NAME}")
print(f"Dave5 parameter values found: {len(baseline_values)}")
print(f"Fixed candidate values:       {len(fixed_candidate_values)}")

print("\nScientific overrides:")
for parameter, value in SCIENTIFIC_OVERRIDES.items():
    print(f"  {parameter}: {value}")

### 7.4.1. Audit parameter sources

Build a review table showing whether each candidate value comes from Optuna, the baseline run, a scientific override, or the unchanged template.

In [ ]:
# ============================================================
# AUDIT CANDIDATE PARAMETER SOURCES
# ============================================================

unknown_overrides = (
    set(SCIENTIFIC_OVERRIDES)
    - set(input_file_parameters)
)

if unknown_overrides:
    raise ValueError(
        "Scientific overrides not found in either input file: "
        f"{sorted(unknown_overrides)}"
    )

audit_rows = []

# dict.fromkeys removes duplicates while preserving order
for parameter in dict.fromkeys(input_file_parameters):

    if parameter in optics_parameters:
        input_file = "Optics"
    else:
        input_file = "Sediment"

    baseline_value = baseline_values.get(
        parameter,
        np.nan
    )

    override_value = SCIENTIFIC_OVERRIDES.get(
        parameter,
        np.nan
    )

    if parameter in active_parameters:

        source = "Optuna"
        candidate_value = "Varies by Optuna"

    elif parameter in SCIENTIFIC_OVERRIDES:

        source = "Scientific override"
        candidate_value = SCIENTIFIC_OVERRIDES[parameter]

    elif parameter in baseline_values:

        source = BASELINE_RUN_NAME
        candidate_value = baseline_values[parameter]

    else:

        source = "Current template"
        candidate_value = "Unchanged"

    audit_rows.append({
        "Parameter": parameter,
        "Input File": input_file,
        "Baseline Value": baseline_value,
        "Scientific Override": override_value,
        "Candidate Source": source,
        "Candidate Value": candidate_value
    })

for parameter in special_optics_parameters:
    audit_rows.append({
        "Parameter": parameter,
        "Input File": "Optics (special block)",
        "Baseline Value": baseline_run.get(parameter, np.nan),
        "Scientific Override": np.nan,
        "Candidate Source": "Optuna",
        "Candidate Value": "Varies by Optuna",
    })

DF_parameter_audit = pd.DataFrame(audit_rows)

display(DF_parameter_audit)

### 7.4.2. Read default parameter values from templates

Parse the current template values without modifying either template. Indexed `nspc` names select one value from a multi-value input line.

In [ ]:
# ============================================================
# READ A PARAMETER VALUE FROM AN INPUT-FILE TEMPLATE
# ============================================================

def read_template_parameter(text, parameter):

    # Handle array values such as bUmax_nspc1
    if "_nspc" in parameter:
        base_parameter, index_text = parameter.split("_nspc")
        value_index = int(index_text)
    else:
        base_parameter = parameter
        value_index = 0

    pattern = (
        rf"^\s*{re.escape(base_parameter)}"
        rf"\s*==\s*([^!\n]+)"
    )

    match = re.search(
        pattern,
        text,
        flags=re.MULTILINE
    )

    if match is None:
        return np.nan

    values_text = match.group(1)

    number_pattern = (
        r"[-+]?"
        r"(?:\d+(?:\.\d*)?|\.\d+)"
        r"(?:[dDeE][-+]?\d+)?"
    )

    number_strings = re.findall(
        number_pattern,
        values_text
    )

    if value_index >= len(number_strings):
        return np.nan

    value_text = (
        number_strings[value_index]
        .replace("D", "e")
        .replace("d", "e")
    )

    return float(value_text)

In [ ]:
# ============================================================
# EXTRACT VALUES FROM CURRENT TEMPLATES
# ============================================================

current_optics_text = OPTICS_TEMPLATE.read_text()
current_sediment_text = SEDIMENT_TEMPLATE.read_text()

current_template_values = {}

for parameter in optics_parameters:

    current_template_values[parameter] = (
        read_template_parameter(
            current_optics_text,
            parameter
        )
    )

for parameter in sediment_parameters:

    current_template_values[parameter] = (
        read_template_parameter(
            current_sediment_text,
            parameter
        )
    )

print(
    "Current template values found:",
    sum(
        pd.notna(value)
        for value in current_template_values.values()
    )
)

### 7.4.3. Compare baseline and template values

Highlight parameters for which the selected baseline and current template disagree, then report which source will control the generated candidate.

In [ ]:
# ============================================================
# PARAMETERS THAT DIFFER BETWEEN CURRENT TEMPLATE AND DAVE5
# ============================================================

different_parameter_rows = []

for parameter in input_file_parameters:

    current_value = current_template_values.get(
        parameter,
        np.nan
    )

    baseline_value = baseline_values.get(
        parameter,
        np.nan
    )

    # A comparison requires both values
    if pd.isna(current_value) or pd.isna(baseline_value):
        continue

    if not np.isclose(
        current_value,
        baseline_value,
        rtol=1e-9,
        atol=1e-12
    ):

        difference = baseline_value - current_value

        if current_value != 0:
            percent_difference = (
                difference / abs(current_value)
            ) * 100
        else:
            percent_difference = np.nan

        if parameter in SCIENTIFIC_OVERRIDES:
            planned_source = "Scientific override"
            planned_value = SCIENTIFIC_OVERRIDES[parameter]

        elif parameter in active_parameters:
            planned_source = "Optuna"
            planned_value = "Varies by Optuna"

        else:
            planned_source = BASELINE_RUN_NAME
            planned_value = baseline_value

        different_parameter_rows.append({
            "Parameter": parameter,
            "Current Template": current_value,
            "Baseline": baseline_value,
            "Difference": difference,
            "Percent Difference": percent_difference,
            "Planned Source": planned_source,
            "Planned Candidate Value": planned_value
        })

DF_different_parameters = pd.DataFrame(
    different_parameter_rows
)

if not DF_different_parameters.empty:

    DF_different_parameters = (
        DF_different_parameters
        .sort_values(
            "Percent Difference",
            key=lambda values: values.abs(),
            ascending=False,
            na_position="last"
        )
        .reset_index(drop=True)
    )

print(
    "Parameters that differ between the current "
    f"templates and {BASELINE_RUN_NAME}: "
    f"{len(DF_different_parameters)}"
)

display(DF_different_parameters)

### 7.4.4. Fixed parameters

Separate fixed values by destination file so each generation loop updates only parameters belonging to that template.

In [ ]:
fixed_optics_parameters = [
    parameter
    for parameter in fixed_candidate_values
    if parameter in optics_parameters
]

fixed_sediment_parameters = [
    parameter
    for parameter in fixed_candidate_values
    if parameter in sediment_parameters
]

print("Fixed optics parameters:")
print(fixed_optics_parameters)

print("\nFixed sediment parameters:")
print(fixed_sediment_parameters)

### 7.4.5. Varying parameters

Preview the first available candidate's Optuna-controlled values before generating text.

In [ ]:
if DF_candidates.empty:
    print("No RUNNING candidates to preview.")
else:
    candidate = DF_candidates.iloc[0]

    print(f"Run Name: {candidate['Run Name']}")
    print("\nOptics parameters:")
    for parameter in active_optics_parameters:
        print(f"{parameter}: {candidate[parameter]}")

    print("\nSediment parameters:")
    for parameter in active_sediment_parameters:
        print(f"{parameter}: {candidate[parameter]}")

    print("\nRepresentative biological-tracer settings:")
    for parameter in special_optics_parameters:
        print(f"{parameter}: {candidate[parameter]}")


## 7.5. Generate water-column input text

For every available candidate, start from a fresh optics template, apply fixed values and Optuna suggestions, update all biological-tracer mixing/advection entries, and point `BSEDPARNAM` to that candidate's sediment file.

In [ ]:
# ============================================================
# GENERATE OPTICS FILE TEXT FOR ALL OPTUNA CANDIDATES
# ============================================================

if prepare_candidate_input_files:

    all_optics_text = {}

    for _, candidate in DF_candidates.iterrows():

        run_name = candidate["Run Name"]

        # Start fresh from the original template
        optics_text = OPTICS_TEMPLATE.read_text()

        # ----------------------------------------------------
        # Apply fixed baseline and scientific overrides
        # ----------------------------------------------------

        for parameter in fixed_optics_parameters:

            new_value = fixed_candidate_values[parameter]

            pattern = (
                rf"^(\s*{re.escape(parameter)}"
                rf"\s*==\s*)([^\s!]+)"
            )

            optics_text, n = re.subn(
                pattern,
                rf"\g<1>{format_roms_value(new_value)}",
                optics_text,
                count=1,
                flags=re.MULTILINE
            )

            if n != 1:
                raise ValueError(
                    f"{run_name}: {parameter} had "
                    f"{n} fixed-value replacements"
                )

        # ----------------------------------------------------
        # Apply Optuna-suggested values
        # ----------------------------------------------------

        for parameter in active_optics_parameters:

            new_value = float(candidate[parameter])

            pattern = (
                rf"^(\s*{re.escape(parameter)}"
                rf"\s*==\s*)([^\s!]+)"
            )

            optics_text, n = re.subn(
                pattern,
                rf"\g<1>{format_roms_value(new_value)}",
                optics_text,
                count=1,
                flags=re.MULTILINE
            )

            if n != 1:
                raise ValueError(
                    f"{run_name}: {parameter} had "
                    f"{n} Optuna-value replacements"
                )

        # ----------------------------------------------------
        # Apply representative biological-tracer settings
        # uniformly to all 15 biological tracers
        # ----------------------------------------------------

        optics_text = replace_biological_tnu2(
            optics_text,
            float(candidate[BIO_NL_TNU2_PARAMETER]),
        )

        advection_category = candidate[BIO_ADVECTION_PARAMETER]
        if advection_category not in BIO_ADVECTION_SCHEMES:
            raise ValueError(
                f"{run_name}: unknown advection category {advection_category!r}"
            )

        horizontal_name, vertical_name = BIO_ADVECTION_SCHEMES[advection_category]
        horizontal_code = ADVECTION_INPUT_CODES[horizontal_name]
        vertical_code = ADVECTION_INPUT_CODES[vertical_name]

        optics_text = replace_advection_block(
            optics_text, "Hadvection", horizontal_code, BIOLOGICAL_TRACER_COUNT
        )
        optics_text = replace_advection_block(
            optics_text, "Vadvection", vertical_code, BIOLOGICAL_TRACER_COUNT
        )

        # ----------------------------------------------------
        # Update BSEDPARNAM
        # ----------------------------------------------------

        sediment_filename = (
            f"bio_UMAINE15_sedmodel_{run_name}.in"
        )

        sediment_path = (
            "/home/abaskind_uri_edu/"
            "ROMS_forcing_files/input_files/OPTUNA/"
            f"{sediment_filename}"
        )

        pattern = r"^(\s*BSEDPARNAM\s*==\s*)(\S+)"

        optics_text, n = re.subn(
            pattern,
            rf"\g<1>{sediment_path}",
            optics_text,
            count=1,
            flags=re.MULTILINE
        )

        if n != 1:
            raise ValueError(
                f"{run_name}: BSEDPARNAM had "
                f"{n} replacements"
            )

        all_optics_text[run_name] = optics_text

        print(f"Finished {run_name}")

### 7.5.1. Inspect generated water-column text

Print the changed scalar lines and complete biological mixing/advection blocks for the first candidate. This is a review step; it does not write files.

In [ ]:
if prepare_candidate_input_files:
    run_name = DF_candidates.iloc[0]["Run Name"]
    lines = all_optics_text[run_name].splitlines()

    for index, line in enumerate(lines):
        if any(line.strip().startswith(parameter) for parameter in active_optics_parameters):
            print(line)
        if line.strip().startswith("TNU2"):
            print(line)
        if line.strip().startswith("Hadvection"):
            print("\n".join(lines[index:index + BIOLOGICAL_TRACER_COUNT]))
        if line.strip().startswith("Vadvection"):
            print("\n".join(lines[index:index + BIOLOGICAL_TRACER_COUNT]))
        if line.strip().startswith("BSED"):
            print(line)

## 7.6. Generate sediment input text

For every available candidate, start from a fresh sediment template and apply fixed and Optuna-suggested values. Scalar parameters and indexed `nspc` values use separate validated replacement paths.

### 7.6.1. Replacement function

Replace either one scalar value or one indexed value on a multi-value sediment line. Missing parameters, invalid indices, and non-finite values raise errors.

In [ ]:
def replace_sediment_parameter(text, parameter, value):

    if not np.isfinite(value):
        raise ValueError(f"{parameter} has non-finite value {value}")

    # --------------------------------
    # Multi-valued parameter
    # Example: bUmax_nspc1
    # --------------------------------

    if "_nspc" in parameter:

        base_parameter, index_text = parameter.split("_nspc")

        index = int(index_text)

        pattern = rf"^(\s*{re.escape(base_parameter)}\s*==\s*)([^!\n]+)(.*)$"

        match = re.search(
            pattern,
            text,
            flags=re.MULTILINE
        )

        if match is None:
            raise ValueError(
                f"Could not find {base_parameter}"
            )

        prefix = match.group(1)
        values_text = match.group(2)
        suffix = match.group(3)

        values = values_text.split()

        if index >= len(values):
            raise ValueError(
                f"{parameter} requests index {index}, but "
                f"{base_parameter} contains {len(values)} value(s)"
            )

        values[index] = format_roms_value(value)

        replacement = (
            prefix
            + " ".join(values)
            + suffix
        )

        text = (
            text[:match.start()]
            + replacement
            + text[match.end():]
        )

    # --------------------------------
    # Scalar parameter
    # Example: bnit
    # --------------------------------

    else:

        pattern = rf"^(\s*{re.escape(parameter)}\s*==\s*)([^\s!]+)"

        text, n = re.subn(
            pattern,
            rf"\g<1>{format_roms_value(value)}",
            text,
            count=1,
            flags=re.MULTILINE
        )

        if n != 1:
            raise ValueError(
                f"{parameter} had {n} replacements"
            )

    return text

### 7.6.2. Generate candidate text

Create and retain one fully modified sediment-text string per candidate.

In [ ]:
# ============================================================
# GENERATE SEDIMENT FILE TEXT FOR ALL OPTUNA CANDIDATES
# ============================================================

if prepare_candidate_input_files:

    all_sediment_text = {}

    for _, candidate in DF_candidates.iterrows():

        run_name = candidate["Run Name"]

        # Start fresh from the original sediment template
        sediment_text = SEDIMENT_TEMPLATE.read_text()

        # ----------------------------------------------------
        # Apply fixed baseline and scientific overrides
        # ----------------------------------------------------

        for parameter in fixed_sediment_parameters:

            sediment_text = replace_sediment_parameter(
                sediment_text,
                parameter,
                fixed_candidate_values[parameter]
            )

        # ----------------------------------------------------
        # Apply Optuna-suggested values
        # ----------------------------------------------------

        for parameter in active_sediment_parameters:

            sediment_text = replace_sediment_parameter(
                sediment_text,
                parameter,
                float(candidate[parameter])
            )

        all_sediment_text[run_name] = sediment_text

        print(f"Finished {run_name}")

### 7.6.3. Inspect generated sediment text

Print key benthic parameter lines from the first generated candidate for review.

In [ ]:
if prepare_candidate_input_files:
    run_name = DF_candidates.iloc[0]["Run Name"]

    for line in all_sediment_text[run_name].splitlines():

        if (
            line.strip().startswith("bUmax")
            or line.strip().startswith("bnit")
            or line.strip().startswith("bdenit")
        ):
            print(line)

## 7.7. Write candidate files

Writing is explicit and occurs only after both generated texts have passed their replacement-count checks. Numeric and advection block replacements must each pass exact-count validation first.

<div class="alert alert-danger"><strong>Overwrite warning:</strong> Existing files with the same candidate names are replaced when <code>WRITE_CANDIDATE_INPUT_FILES</code> is enabled.</div>

In [ ]:
# ============================================================
# WRITE MODEL INPUT FILES FOR ALL OPTUNA CANDIDATES
# ============================================================
if prepare_candidate_input_files:

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    generated_files = []

    for _, candidate in DF_candidates.iterrows():

        run_name = candidate["Run Name"]

        # --------------------------------------------------------
        # File names
        # --------------------------------------------------------

        optics_filename = (
            f"bio_UMAINE15_sediment_optics_{run_name}.in"
        )

        sediment_filename = (
            f"bio_UMAINE15_sedmodel_{run_name}.in"
        )

        optics_output = OUTPUT_DIR / optics_filename
        sediment_output = OUTPUT_DIR / sediment_filename

        # --------------------------------------------------------
        # Write the already-modified text
        # --------------------------------------------------------

        optics_output.write_text(
            all_optics_text[run_name]
        )

        sediment_output.write_text(
            all_sediment_text[run_name]
        )

        # --------------------------------------------------------
        # Keep a record
        # --------------------------------------------------------

        generated_files.append({
            "Run Name": run_name,
            "Optics File": optics_filename,
            "Sediment File": sediment_filename
        })

        print(f"Wrote {run_name}")

### 7.7.1. Verify written files

List the generated file manifest and count matching optics and sediment files in the output directory.

In [ ]:
if prepare_candidate_input_files:
    DF_generated_files = pd.DataFrame(generated_files)

    display(DF_generated_files)

In [ ]:
if prepare_candidate_input_files:
    print("Files written to:")
    print(OUTPUT_DIR)

    print("\nNumber of optics files:")
    print(len(list(OUTPUT_DIR.glob("bio_UMAINE15_sediment_optics_*.in"))))

    print("\nNumber of sediment files:")
    print(len(list(OUTPUT_DIR.glob("bio_UMAINE15_sedmodel_*.in"))))

<a id="final-status"></a>
# ✅ 8. Final study status and export

Review the final trial states after optional synchronization and candidate generation. Remaining `RUNNING` trials are listed by number and run name. The cleaned run summary is written only when `EXPORT_RUN_SUMMARY` is `True`; otherwise it remains available in memory as `DF_run_summary`.

<div class="alert alert-info"><strong>Export behavior:</strong> Building <code>DF_run_summary</code> is read-only. A CSV is written only when <code>EXPORT_RUN_SUMMARY=True</code>.</div>

In [25]:
# ============================================================
# FINAL STUDY STATUS
# ============================================================

print(f"Study: {study.study_name}")
print(f"Total trials: {len(study.trials)}")

for state in optuna.trial.TrialState:
    count = sum(trial.state == state for trial in study.trials)
    if count > 0:
        print(f"  {state.name}: {count}")

Study: model_calibration_bio
Total trials: 106
  COMPLETE: 106


In [26]:
running_trials = [
    trial
    for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]

print(f"\nTrials still RUNNING: {len(running_trials)}")

for trial in running_trials:
    print(
        f"  Trial {trial.number}: "
        f"{trial.user_attrs.get('Run Name')}"
    )


Trials still RUNNING: 0


In [27]:
# ============================================================
# EXPORT VALIDATED RUN SUMMARY FOR PARAMETER ANALYSIS
# ============================================================

# >>> USER INPUT (optional): Change the export filename if a separate
# version should be retained. Writing still requires EXPORT_RUN_SUMMARY=True.
RUN_SUMMARY_FILE = OPT_DIR / "Model_Run_Summary_Bio.csv"

DF_run_summary = DF_opt.copy()

DF_run_summary["Objective Version"] = OBJECTIVE_VERSION

# Match each run to its actual Optuna trial number
trial_number_by_run = {
    trial.user_attrs.get("Run Name"): trial.number
    for trial in study.trials
    if trial.user_attrs.get("Run Name") is not None
}

DF_run_summary["Trial Number"] = (
    DF_run_summary["Run Name"]
    .map(trial_number_by_run)
    .astype("Int64")
)

if EXPORT_RUN_SUMMARY:
    DF_run_summary.to_csv(
        RUN_SUMMARY_FILE,
        index=False,
    )
    print("Saved validated model-run summary:")
    print(RUN_SUMMARY_FILE)
else:
    print("Run-summary export is OFF.")

print(f"\nRows:    {len(DF_run_summary)}")
print(f"Columns: {len(DF_run_summary.columns)}")
print(
    "Runs matched to Optuna trials:",
    DF_run_summary["Trial Number"].notna().sum()
)

Saved validated model-run summary:
/Users/akbaskind/Desktop/Optimization/Model_Run_Summary_Bio.csv

Rows:    107
Columns: 130
Runs matched to Optuna trials: 106
